# Обучающая выборка

Цель — найти короткий непрерывный блок обучающей выборки, на котором все модели обучаются так же хорошо, как на всей выборке: обучение и выбор порога идут быстрее, условия у всех моделей одинаковые.

Ячейки до раздела «Поиск блока» — копии ячеек example_model.ipynb (коммит 3a238c1): константы, импорты, ENU, предобработка, метрики, общие функции, функции пяти моделей без циклов по сценариям. При изменении этих ячеек в основном блокноте копии обновляются.

In [ ]:
TORCH_CUDA_INDEX = "https://download.pytorch.org/whl/cu124"
ALLOWED_STATUS = ("move", "anomaly")

TEST_SPLIT_1 = 0.6
TEST_SPLIT_2 = 0.8
MCC_BAND = 0.02
GRID_STEP_SEC = 10.0
TIME_GAP_SEC = 180.0
THRESHOLD_LOG_POINTS = 60
PLOT_TAIL = 50_000
MAP_FIGSIZE = (22.0, 11.0)
# True — строятся все карты; False — только текст и таблицы, в том числе таблицы метрик после каждой модели.
SHOW_PLOTS = False
N_WINDOWS = 8_000
# Нейросети (LSTM, TCN, DONUT): ранняя остановка по ошибке на калибровочном блоке — последние окна по времени.
NN_MAX_EPOCHS = 300
NN_PATIENCE = 10
NN_MIN_DELTA = 1e-4
NN_LR_PATIENCE = 3
NN_LR_FACTOR = 0.5
NN_CALIB_FRACTION = 0.15
TORCH_BATCH_SIZE = 256
TORCH_INFER_BATCH = 512
TORCH_LR = 1e-3

In [ ]:
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path
from typing import Callable, Optional


def ensure_cuda_torch(index_url: str) -> None:
    """Ставит сборку torch с CUDA, если есть GPU NVIDIA, а установленный torch собран без CUDA.

    Вызывается до первого импорта torch в ядре: уже загруженный torch в процессе не заменить. Сборка torch
    проверяется в отдельном процессе, чтобы не загружать его в ядро раньше времени.

    Args:
        index_url: Индекс колёс PyTorch с нужной версией CUDA (``TORCH_CUDA_INDEX``).
    """
    if shutil.which("nvidia-smi") is None:
        return
    probe = subprocess.run(
        [sys.executable, "-c", "import torch; print(torch.version.cuda or '')"],
        capture_output=True,
        text=True,
        check=False,
    )
    if probe.returncode == 0 and probe.stdout.strip():
        return
    if "torch" in sys.modules:
        print("torch без CUDA уже загружен в этом ядре: перезапустите ядро и выполните блокнот заново")
        return
    print(f"Найден GPU NVIDIA, torch собран без CUDA: установка сборки с CUDA из {index_url}")
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torch"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch", "--index-url", index_url], check=True)


# До импорта torch: при GPU NVIDIA ставится CUDA-сборка, иначе остаётся текущая и ниже выбирается CPU.
ensure_cuda_torch(TORCH_CUDA_INDEX)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from hmmlearn import _hmmc
from IPython.display import display
from numpy.linalg import LinAlgError
from scipy.linalg import solve_triangular
from scipy.optimize import linprog, minimize
from scipy.special import logsumexp
from scipy.stats import chi2
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import NearestNeighbors
from sklearn.svm import OneClassSVM
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.kalman_filter import KalmanFilter
from statsmodels.tsa.statespace.kalman_smoother import KalmanSmoother
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

# Блокнот запускается из notebooks/: пакет app импортируется из корня проекта.
path_root = Path.cwd().parent
if str(path_root) not in sys.path:
    sys.path.insert(0, str(path_root))

from app.evaluation import TrajectoryMetrics, TrajectorySeries
from app.viz import plot_track_maps

path_data = path_root / "data" / "2.csv"

get_ipython().run_line_magic("matplotlib", "inline")

plt.rcParams["font.family"] = "DejaVu Sans"
plt.rcParams["axes.unicode_minus"] = False

SEED = 42
rng = np.random.default_rng(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    DEVICE = torch.device("cuda:0")
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
else:
    DEVICE = torch.device("cpu")
torch.set_num_threads(max(1, min(8, os.cpu_count() or 1)))

print(f"Корень: {path_root}")
print(f"Файл: {path_data}")
print(f"PyTorch {torch.__version__}, устройство {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")

In [ ]:
_WGS84_A_M = 6_378_137.0
_WGS84_F = 1.0 / 298.257223563
_WGS84_E2 = 2.0 * _WGS84_F - _WGS84_F**2


def _validate_wgs84_coordinates(
    lat_deg: np.ndarray,
    lon_deg: np.ndarray,
) -> None:
    """Проверяет форму и допустимые диапазоны широты и долготы.

    Args:
        lat_deg: Широты, градусы.
        lon_deg: Долготы, градусы.

    Raises:
        ValueError: Если формы не совпадают или координаты выходят за пределы WGS-84.
    """
    if lat_deg.shape != lon_deg.shape:
        raise ValueError("Массивы широт и долгот должны иметь одинаковую форму")
    if not np.all(np.isfinite(lat_deg)) or not np.all(np.isfinite(lon_deg)):
        raise ValueError("Координаты должны быть конечными числами")
    if np.any((lat_deg < -90.0) | (lat_deg > 90.0)):
        raise ValueError("Широта должна лежать в диапазоне [-90, 90]")
    if np.any((lon_deg < -180.0) | (lon_deg > 180.0)):
        raise ValueError("Долгота должна лежать в диапазоне [-180, 180]")


def wgs84_to_ecef(
    lat_deg: np.ndarray,
    lon_deg: np.ndarray,
    height_m: float | np.ndarray = 0.0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Переводит геодезические координаты WGS-84 в ECEF.

    Args:
        lat_deg: Широта, градусы.
        lon_deg: Долгота, градусы.
        height_m: Высота над эллипсоидом, м; по умолчанию 0.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: X, Y, Z в метрах.

    Raises:
        ValueError: Если координаты имеют разную форму или выходят за пределы WGS-84.
    """
    lat = np.asarray(lat_deg, dtype=np.float64)
    lon = np.asarray(lon_deg, dtype=np.float64)
    _validate_wgs84_coordinates(lat, lon)
    height = np.broadcast_to(np.asarray(height_m, dtype=np.float64), lat.shape)
    phi = np.deg2rad(lat)
    lam = np.deg2rad(lon)
    sin_phi = np.sin(phi)
    cos_phi = np.cos(phi)
    prime_vertical = _WGS84_A_M / np.sqrt(1.0 - _WGS84_E2 * sin_phi**2)
    radius = prime_vertical + height
    x_ecef = radius * cos_phi * np.cos(lam)
    y_ecef = radius * cos_phi * np.sin(lam)
    z_ecef = (prime_vertical * (1.0 - _WGS84_E2) + height) * sin_phi
    return x_ecef, y_ecef, z_ecef


def wgs84_to_enu(
    lat_deg: np.ndarray,
    lon_deg: np.ndarray,
    origin_lat_deg: float,
    origin_lon_deg: float,
    height_m: float | np.ndarray = 0.0,
    origin_height_m: float = 0.0,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Переводит координаты WGS-84 в локальную ENU относительно опорной точки.

    Args:
        lat_deg: Широты точек, градусы.
        lon_deg: Долготы точек, градусы.
        origin_lat_deg: Широта опорной точки, градусы.
        origin_lon_deg: Долгота опорной точки, градусы.
        height_m: Высота точек над эллипсоидом, м.
        origin_height_m: Высота опорной точки, м.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: E, N, U в метрах.
    """
    origin_lat = float(origin_lat_deg)
    origin_lon = float(origin_lon_deg)
    origin_height = float(origin_height_m)
    x_ecef, y_ecef, z_ecef = wgs84_to_ecef(lat_deg, lon_deg, height_m)
    x0, y0, z0 = wgs84_to_ecef(origin_lat, origin_lon, origin_height)
    dx = x_ecef - x0
    dy = y_ecef - y0
    dz = z_ecef - z0
    phi0 = np.deg2rad(origin_lat)
    lam0 = np.deg2rad(origin_lon)
    sin_phi0 = np.sin(phi0)
    cos_phi0 = np.cos(phi0)
    sin_lam0 = np.sin(lam0)
    cos_lam0 = np.cos(lam0)
    east = -sin_lam0 * dx + cos_lam0 * dy
    north = -sin_phi0 * cos_lam0 * dx - sin_phi0 * sin_lam0 * dy + cos_phi0 * dz
    up = cos_phi0 * cos_lam0 * dx + cos_phi0 * sin_lam0 * dy + sin_phi0 * dz
    return east, north, up


def wgs84_to_local_xy(
    lat_deg: np.ndarray,
    lon_deg: np.ndarray,
    origin_lat_deg: float,
    origin_lon_deg: float,
) -> tuple[np.ndarray, np.ndarray]:
    """Переводит WGS-84 в локальные x (восток) и y (север), метры.

    Args:
        lat_deg: Широты точек, градусы.
        lon_deg: Долготы точек, градусы.
        origin_lat_deg: Широта опорной точки, градусы.
        origin_lon_deg: Долгота опорной точки, градусы.

    Returns:
        tuple[np.ndarray, np.ndarray]: x (восток) и y (север) в метрах.
    """
    east, north, _ = wgs84_to_enu(lat_deg, lon_deg, origin_lat_deg, origin_lon_deg)
    return east, north

In [ ]:
def nfmt(value: int) -> str:
    """Форматирует целое число с разделителем тысяч (пробел).

    Args:
        value: целое значение для вывода.

    Returns:
        str: строка вида «1 234 567».
    """
    return f"{int(value):,}".replace(",", " ")


def fmt_num(value: float, decimals: int = 2) -> str:
    """Число; при |value| >= 1000 — разделитель тысяч «_».

    Args:
        value: Значение для вывода.
        decimals: Число знаков после запятой (0 — целое).

    Returns:
        str: Отформатированная строка или «—» для нечисловых значений.
    """
    if value is None or not np.isfinite(float(value)):
        return "—"
    number = float(value)
    negative = number < 0
    number = abs(number)
    if number < 1000:
        text = f"{number:.{decimals}f}" if decimals else str(int(round(number)))
        if "." in text:
            text = text.rstrip("0").rstrip(".")
        return ("-" if negative else "") + text
    if decimals == 0:
        body = f"{int(round(number)):,}".replace(",", "_")
    else:
        whole = int(number)
        fraction = number - whole
        fraction_text = f"{fraction:.{decimals}f}".split(".")[1].rstrip("0")
        body = f"{whole:,}".replace(",", "_")
        if fraction_text:
            body = f"{body}.{fraction_text}"
    return ("-" if negative else "") + body


def select_trajectory_columns(raw: pd.DataFrame) -> pd.DataFrame:
    """Оставляет столбцы time, lat, lon и переименовывает status в y.

    Args:
        raw: таблица из data/test.csv.

    Returns:
        pd.DataFrame: столбцы time, lat, lon, y.
    """
    return raw.loc[:, ["time", "lat", "lon", "status"]].rename(columns={"status": "y"}).copy()


def coerce_coordinate_columns(frame: pd.DataFrame) -> pd.DataFrame:
    """Приводит lat и lon к числовому типу (нечисловые значения → NaN).

    Args:
        frame: таблица с столбцами lat, lon.

    Returns:
        pd.DataFrame: копия с числовыми lat, lon.
    """
    result = frame.copy()
    result["lat"] = pd.to_numeric(result["lat"], errors="coerce")
    result["lon"] = pd.to_numeric(result["lon"], errors="coerce")
    return result


def filter_finite_coordinates(frame: pd.DataFrame) -> pd.DataFrame:
    """Удаляет строки с пропусками в lat или lon.

    Args:
        frame: таблица с lat, lon.

    Returns:
        pd.DataFrame: только строки с конечными координатами.
    """
    mask = frame["lat"].notna() & frame["lon"].notna()
    return frame.loc[mask].reset_index(drop=True)


def filter_coordinates_wgs84(frame: pd.DataFrame) -> pd.DataFrame:
    """Удаляет строки с координатами вне допустимых пределов WGS-84.

    Args:
        frame: таблица с lat, lon.

    Returns:
        pd.DataFrame: lat в [-90, 90], lon в [-180, 180].
    """
    mask = frame["lat"].between(-90.0, 90.0) & frame["lon"].between(-180.0, 180.0)
    return frame.loc[mask].reset_index(drop=True)


def filter_nonzero_gps(frame: pd.DataFrame) -> pd.DataFrame:
    """Удаляет точки «нулевого» GPS (lat, lon) = (0, 0).

    Args:
        frame: таблица с lat, lon.

    Returns:
        pd.DataFrame: отфильтрованный ряд.
    """
    mask = ~((frame["lat"] == 0.0) & (frame["lon"] == 0.0))
    return frame.loc[mask].reset_index(drop=True)


def filter_allowed_status(frame: pd.DataFrame, allowed: tuple[str, ...]) -> pd.DataFrame:
    """Оставляет только метки из allowed (move, anomaly).

    Args:
        frame: таблица со столбцом y.
        allowed: допустимые строковые значения status.

    Returns:
        pd.DataFrame: отфильтрованный ряд.
    """
    labels = frame["y"].astype(str)
    return frame.loc[labels.isin(allowed)].reset_index(drop=True)


def sort_by_time(frame: pd.DataFrame) -> pd.DataFrame:
    """Сортирует ряд по возрастанию time (стабильная сортировка).

    Args:
        frame: таблица со столбцом time.

    Returns:
        pd.DataFrame: упорядоченный ряд.
    """
    order = np.argsort(pd.to_datetime(frame["time"]).to_numpy(), kind="mergesort")
    return frame.iloc[order].reset_index(drop=True)


def attach_local_enu(frame: pd.DataFrame) -> pd.DataFrame:
    """Добавляет столбцы x, n — локальная ENU (м) от первой точки ряда.

    Args:
        frame: таблица с lat, lon.

    Returns:
        pd.DataFrame: с добавленными x, n.
    """
    result = frame.copy()
    lat = result["lat"].to_numpy(dtype=float)
    lon = result["lon"].to_numpy(dtype=float)
    east, north = wgs84_to_local_xy(lat, lon, lat[0], lon[0])
    result["x"] = east
    result["n"] = north
    return result


def trim_to_extreme_move(frame: pd.DataFrame) -> pd.DataFrame:
    """Обрезает ряд от первой до последней точки со статусом move.

    Args:
        frame: таблица со столбцом y.

    Returns:
        pd.DataFrame: срез между крайними move.

    Raises:
        RuntimeError: если в ряду нет точек move.
    """
    is_move = frame["y"].astype(str).to_numpy() == "move"
    if not np.any(is_move):
        raise RuntimeError("После фильтрации нет точек move.")
    move_idx = np.flatnonzero(is_move)
    return frame.iloc[move_idx[0] : move_idx[-1] + 1].reset_index(drop=True)


def attach_binary_label(frame: pd.DataFrame) -> pd.DataFrame:
    """Добавляет столбец label для TrajectoryMetrics / TrajectorySeries.

    Положительный класс в метриках (MCC, precision, recall) — anomaly. В этом
    проекте он кодируется нулём (не единицей): anomaly → 0, move → 1. См.
    anomaly_mask (label == 0).

    Args:
        frame: таблица со столбцом y (строки move / anomaly).

    Returns:
        pd.DataFrame: с столбцом label (float).
    """
    result = frame.copy()
    result["label"] = np.where(result["y"].astype(str) == "anomaly", 0.0, 1.0)
    return result


def split_train_test_move(frame: pd.DataFrame, train_fraction: float) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Делит ряд на обучение [0, k) и тест [k, N); граница k — первый move с индексом i >= floor(rho*N).

    Args:
        frame: предобработанный ряд с столбцом y (move / anomaly).
        train_fraction: доля обучения rho из (0, 1), как TEST_SPLIT_1 или TEST_SPLIT_2.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame]: (train, test).

    Raises:
        ValueError: если train_fraction не в (0, 1).
        RuntimeError: если ряд слишком короткий, нет подходящего move или выборка пуста.
    """
    if not 0.0 < train_fraction < 1.0:
        raise ValueError("train_fraction должен быть в (0, 1).")
    n = len(frame)
    if n < 2:
        raise RuntimeError("Слишком короткий ряд для разбиения.")
    k0 = int(np.floor(train_fraction * n))
    k0 = min(max(k0, 1), n - 1)
    labels = frame["y"].astype(str).to_numpy()
    move_from_k0 = np.flatnonzero((np.arange(n) >= k0) & (labels == "move"))
    if move_from_k0.size == 0:
        raise RuntimeError(f"Нет move для границы при rho={train_fraction}.")
    k = int(move_from_k0[0])
    train = frame.iloc[:k].reset_index(drop=True)
    test = frame.iloc[k:].reset_index(drop=True)
    if len(train) == 0 or len(test) == 0:
        raise RuntimeError(f"Пустая выборка при rho={train_fraction}, k={k}, N={n}.")
    return train, test


def describe_split(name: str, train: pd.DataFrame, test: pd.DataFrame) -> None:
    """Печатает размер и доли классов anomaly и move для обучения и теста.

    Args:
        name: заголовок сценария (например, df1 и доля rho).
        train: обучающая выборка.
        test: тестовая выборка.
    """
    for part_name, part in (("обучение", train), ("тест", test)):
        labels = part["y"].astype(str)
        n = len(part)
        p_anom = float((labels == "anomaly").mean())
        p_move = float((labels == "move").mean())
        print(
            f"{name}: {part_name:<9} {nfmt(n):>10} точек   "
            f"anomaly {p_anom:6.1%}   move {p_move:6.1%}"
        )


raw_full = pd.read_csv(path_data)
frame = select_trajectory_columns(raw_full)
frame = coerce_coordinate_columns(frame)
frame = filter_finite_coordinates(frame)
frame = filter_coordinates_wgs84(frame)
frame = filter_nonzero_gps(frame)
frame = filter_allowed_status(frame, ALLOWED_STATUS)
frame = sort_by_time(frame)
frame = attach_local_enu(frame)
frame = trim_to_extreme_move(frame)
frame = attach_binary_label(frame)
train_1, test_1 = split_train_test_move(frame, TEST_SPLIT_1)
train_2, test_2 = split_train_test_move(frame, TEST_SPLIT_2)

df1 = {"train": train_1, "test": test_1}
df2 = {"train": train_2, "test": test_2}

print(f"Исходный CSV: {nfmt(len(raw_full))} строк, после предобработки: {nfmt(len(frame))}.")
describe_split("df1 (rho={:.0%})".format(TEST_SPLIT_1), train_1, test_1)
describe_split("df2 (rho={:.0%})".format(TEST_SPLIT_2), train_2, test_2)

In [ ]:
def confusion_counts(
    predicted_anomaly: np.ndarray,
    ground_truth_anomaly: np.ndarray,
) -> tuple[float, float, float, float]:
    """Считает TP, FP, FN, TN для масок anomaly.

    Положительный класс — anomaly (в исходной разметке label == 0).

    Args:
        predicted_anomaly: Булева маска предсказанных anomaly.
        ground_truth_anomaly: Булева маска эталонных anomaly.

    Returns:
        tuple[float, float, float, float]: (TP, FP, FN, TN).

    Raises:
        ValueError: Если длины масок не совпадают.
    """
    predicted = np.asarray(predicted_anomaly, dtype=bool).ravel()
    truth = np.asarray(ground_truth_anomaly, dtype=bool).ravel()
    if predicted.shape != truth.shape:
        raise ValueError("Маски предсказания и эталона должны иметь одинаковую длину.")
    return (
        float(np.count_nonzero(predicted & truth)),
        float(np.count_nonzero(predicted & ~truth)),
        float(np.count_nonzero(~predicted & truth)),
        float(np.count_nonzero(~predicted & ~truth)),
    )


def mcc_from_counts(
    true_positive: float,
    false_positive: float,
    false_negative: float,
    true_negative: float,
) -> float:
    """Считает MCC по элементам таблицы сопряжённости.

    Args:
        true_positive: TP — anomaly верно предсказана.
        false_positive: FP — anomaly предсказана на точке move.
        false_negative: FN — anomaly пропущена.
        true_negative: TN — move верно оставлен move.

    Returns:
        float: MCC в [-1, 1]; 0.0, если знаменатель равен нулю.
    """
    denominator = np.sqrt(
        (true_positive + false_positive)
        * (true_positive + false_negative)
        * (true_negative + false_positive)
        * (true_negative + false_negative)
    )
    if denominator <= 0.0:
        return 0.0
    numerator = true_positive * true_negative - false_positive * false_negative
    return float(numerator / denominator)


def mcc_score(
    predicted_anomaly: np.ndarray,
    ground_truth_anomaly: np.ndarray,
) -> float:
    """Считает MCC по двум маскам anomaly.

    Args:
        predicted_anomaly: Булева маска предсказанных anomaly.
        ground_truth_anomaly: Булева маска эталонных anomaly.

    Returns:
        float: MCC в [-1, 1]; 0.0 при вырожденной таблице сопряжённости.
    """
    true_positive, false_positive, false_negative, true_negative = confusion_counts(
        predicted_anomaly, ground_truth_anomaly
    )
    return mcc_from_counts(true_positive, false_positive, false_negative, true_negative)

In [ ]:
def trajectory_series_pair(
    frame: pd.DataFrame, predicted_anomaly: np.ndarray, scored: Optional[np.ndarray] = None
) -> tuple[TrajectorySeries, TrajectorySeries]:
    """Собирает prediction и ground truth для TrajectoryMetrics.

    Args:
        frame: Строки test с полями time, lat, lon, y.
        predicted_anomaly: True, если модель объявила anomaly.
        scored: True в точках с оценкой аномальности; None — оценка есть во всех точках.

    Returns:
        tuple[TrajectorySeries, TrajectorySeries]: (prediction, ground_truth); в prediction точка без оценки — NaN.
    """
    ground_anomaly = frame["y"].astype(str).to_numpy() == "anomaly"
    truth = np.where(ground_anomaly, 0.0, 1.0)
    predicted = np.where(predicted_anomaly, 0.0, 1.0)
    if scored is not None:
        predicted[~np.asarray(scored, dtype=bool)] = np.nan
    time = frame["time"].to_numpy()
    lon = frame["lon"].to_numpy(dtype=float)
    lat = frame["lat"].to_numpy(dtype=float)
    ground_truth = TrajectorySeries(time, lon, lat, truth)
    prediction = TrajectorySeries(time, lon, lat, predicted)
    return prediction, ground_truth


def slice_trajectory_series(series: TrajectorySeries, tail: int) -> TrajectorySeries:
    """Хвост TrajectorySeries для карты (последние tail точек).

    Args:
        series: Исходный ряд.
        tail: Число точек с конца.

    Returns:
        TrajectorySeries: Срез хвоста.
    """
    count = min(int(tail), len(series.time))
    return TrajectorySeries(
        series.time[-count:],
        series.lon[-count:],
        series.lat[-count:],
        series.label[-count:],
    )


def evaluate_test(
    frame: pd.DataFrame, predicted_anomaly: np.ndarray, scored: Optional[np.ndarray] = None
) -> dict[str, float]:
    """Считает на тесте метрики из Отчёта (TrajectoryMetrics.compute_all).

    Args:
        frame: Тестовый DataFrame.
        predicted_anomaly: Маска предсказанных anomaly.
        scored: True в точках с оценкой аномальности; None — оценка есть во всех точках.

    Returns:
        dict[str, float]: Гео-, временные, классификационные метрики и покрытие test.
    """
    prediction, ground_truth = trajectory_series_pair(frame, predicted_anomaly, scored)
    return TrajectoryMetrics.compute_all(prediction, ground_truth)


def time_seconds_from_frame(frame: pd.DataFrame) -> np.ndarray:
    """Время ряда в секундах от эпохи (float64).

    Args:
        frame: DataFrame с колонкой time, приводимой к datetime.

    Returns:
        np.ndarray: Секунды от эпохи Unix, shape (N,).
    """
    stamp = pd.to_datetime(frame["time"])
    return stamp.to_numpy(dtype="datetime64[ns]").astype(np.int64) / 1e9


def enu_xy_from_frame(frame: pd.DataFrame) -> np.ndarray:
    """Координаты ENU (восток, север), м, форма (N, 2).

    Args:
        frame: DataFrame с колонками x (восток) и n (север), м.

    Returns:
        np.ndarray: Матрица (N, 2) координат ENU, м.
    """
    return np.column_stack([frame["x"].to_numpy(dtype=float), frame["n"].to_numpy(dtype=float)])


def wrap_angle(radians: np.ndarray) -> np.ndarray:
    """Перенос угла в интервал (-pi, pi].

    Args:
        radians: Углы, рад.

    Returns:
        np.ndarray: Те же углы в интервале (-pi, pi].
    """
    return (np.asarray(radians, dtype=float) + np.pi) % (2.0 * np.pi) - np.pi


def trajectory_kinematics(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Скорость, ускорение, поворот курса и Δt; у i=0 нули.

    Args:
        frame: DataFrame с колонками x, n и time.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]: speed (м/с), acc (м/с²),
        dpsi (рад) и step_dt (с).
    """
    east = frame["x"].to_numpy(dtype=float)
    north = frame["n"].to_numpy(dtype=float)
    time_sec = time_seconds_from_frame(frame)
    count = len(east)
    step_dt = np.ones(count, dtype=float)
    speed = np.zeros(count, dtype=float)
    acc = np.zeros(count, dtype=float)
    dpsi = np.zeros(count, dtype=float)
    if count < 2:
        return speed, acc, dpsi, step_dt
    step_dt[1:] = np.diff(time_sec)
    dist = np.hypot(np.diff(east), np.diff(north))
    speed[1:] = dist / np.clip(step_dt[1:], 1.0, None)
    acc[1:] = np.diff(speed) / np.clip(step_dt[1:], 1.0, None)
    heading = np.zeros(count, dtype=float)
    dx_step = east[1:] - east[:-1]
    dy_step = north[1:] - north[:-1]
    heading[1:] = np.arctan2(dx_step, dy_step)
    if count > 2:
        dpsi[2:] = wrap_angle(np.diff(heading[1:]))
    return speed, acc, dpsi, step_dt


def robust_params(matrix: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Робастная шкала: медиана и межквартильный размах по столбцам.

    Args:
        matrix: Матрица признаков shape (n, d).

    Returns:
        tuple[np.ndarray, np.ndarray]: center (медианы) и scale (IQR); scale < 1e-6 → 1.
    """
    center = np.median(matrix, axis=0)
    scale = np.subtract(*np.percentile(matrix, [75, 25], axis=0))
    scale = np.where(scale < 1e-6, 1.0, scale)
    return center, scale


def apply_robust(matrix: np.ndarray, center: np.ndarray, scale: np.ndarray) -> np.ndarray:
    """Центрирование и деление на scale.

    Args:
        matrix: Матрица признаков shape (n, d).
        center: Центры столбцов из robust_params.
        scale: Масштабы столбцов из robust_params.

    Returns:
        np.ndarray: (matrix - center) / scale.
    """
    return (matrix - center) / scale


def frame_window_arrays(frame: pd.DataFrame) -> dict[str, np.ndarray]:
    """Поля для оконных детекторов: ENU, шаги, кинематика, move.

    Args:
        frame: DataFrame с колонками x, n, time и y.

    Returns:
        dict[str, np.ndarray]: count, xy, dx, dy, speed, dpsi, dt, dist, move.
    """
    east = frame["x"].to_numpy(dtype=float)
    north = frame["n"].to_numpy(dtype=float)
    speed, _, dpsi, step_dt = trajectory_kinematics(frame)
    count = len(east)
    step_dx = np.zeros(count, dtype=float)
    step_dy = np.zeros(count, dtype=float)
    step_dist = np.zeros(count, dtype=float)
    if count > 1:
        step_dx[1:] = np.diff(east)
        step_dy[1:] = np.diff(north)
        step_dist[1:] = np.hypot(step_dx[1:], step_dy[1:])
    return {
        "count": count,
        "xy": np.column_stack([east, north]),
        "dx": step_dx,
        "dy": step_dy,
        "speed": speed,
        "dpsi": dpsi,
        "dt": step_dt,
        "dist": step_dist,
        "move": frame["y"].astype(str).to_numpy() == "move",
    }


def window_starts(
    arrays: dict[str, np.ndarray],
    length: int,
    max_dt: float,
    max_dist: float,
    require_move: bool,
) -> np.ndarray:
    """Индексы окон без большого разрыва по времени или расстоянию.

    Args:
        arrays: Кэш ряда из frame_window_arrays.
        length: Длина окна в точках.
        max_dt: Максимальный Δt между соседними точками, с.
        max_dist: Максимальный шаг между соседними точками, м.
        require_move: Требовать move во всех точках окна.

    Returns:
        np.ndarray: Индексы допустимых стартов; пустой массив, если окон нет.
    """
    count = arrays["count"]
    if count < length:
        return np.empty(0, dtype=int)
    good = np.ones(count, dtype=bool)
    if require_move:
        good &= arrays["move"]
    step_ok = np.ones(count, dtype=bool)
    step_ok[1:] = (arrays["dt"][1:] <= max_dt) & (arrays["dist"][1:] <= max_dist)
    good_cum = np.cumsum(good.astype(np.int32))
    step_cum = np.cumsum(step_ok.astype(np.int32))
    starts = np.arange(count - length + 1)
    good_sum = good_cum[starts + length - 1] - np.where(starts > 0, good_cum[starts - 1], 0)
    step_sum = step_cum[starts + length - 1] - step_cum[starts]
    return starts[(good_sum == length) & (step_sum == length - 1)]


def take_window_starts(starts: np.ndarray, limit: int, seed: int) -> np.ndarray:
    """Случайная подвыборка стартов окон без повторов.

    Args:
        starts: Допустимые индексы начала окна.
        limit: Максимальное число стартов.
        seed: Seed генератора.

    Returns:
        np.ndarray: Отсортированные выбранные старты.
    """
    if len(starts) <= limit:
        return starts
    picked = np.random.default_rng(seed).choice(len(starts), size=limit, replace=False)
    return np.sort(starts[picked])


def batched_model_forward(model: nn.Module, array: np.ndarray, forward_fn, batch_size: int) -> np.ndarray:
    """Пакетный forward PyTorch-модели без градиентов.

    Args:
        model: Модель на DEVICE.
        array: Входной numpy-массив.
        forward_fn: Callable (model, batch_tensor) -> tensor.
        batch_size: Размер пакета.

    Returns:
        np.ndarray: Склеенные выходы по оси 0.
    """
    model.eval()
    pieces: list[np.ndarray] = []
    with torch.no_grad():
        for start in range(0, len(array), batch_size):
            batch = torch.from_numpy(np.ascontiguousarray(array[start : start + batch_size])).to(DEVICE)
            pieces.append(forward_fn(model, batch).detach().cpu().numpy())
    return np.concatenate(pieces, axis=0)


def operational_path_length_m(east_m: np.ndarray, north_m: np.ndarray, move_mask: np.ndarray) -> float:
    """Длина ломаной в ENU только по точкам move (без интерполяции).

    Args:
        east_m: Восток, м.
        north_m: Север, м.
        move_mask: True для штатного хода.

    Returns:
        float: Длина в метрах.
    """
    move = np.asarray(move_mask, dtype=bool)
    east = np.asarray(east_m, dtype=np.float64)[move]
    north = np.asarray(north_m, dtype=np.float64)[move]
    if east.size < 2:
        return 0.0
    return float(np.sum(np.hypot(np.diff(east), np.diff(north))))


def operational_distance_loss_m(
    east_m: np.ndarray,
    north_m: np.ndarray,
    ground_truth_anomaly: np.ndarray,
    predicted_anomaly: np.ndarray,
) -> float:
    """Операционный |L_pred - L_ref| по move-точкам в ENU.

    Args:
        east_m: Восток, м.
        north_m: Север, м.
        ground_truth_anomaly: True для anomaly в разметке.
        predicted_anomaly: True для предсказанной anomaly.

    Returns:
        float: Distance loss в метрах.
    """
    gt_anomaly = np.asarray(ground_truth_anomaly, dtype=bool)
    pred_anomaly = np.asarray(predicted_anomaly, dtype=bool)
    reference_m = operational_path_length_m(east_m, north_m, ~gt_anomaly)
    predicted_m = operational_path_length_m(east_m, north_m, ~pred_anomaly)
    return float(abs(predicted_m - reference_m))


def _threshold_record(
    tau: float,
    predicted_anomaly: np.ndarray,
    frame: pd.DataFrame,
    ground_truth_anomaly: np.ndarray,
    scored: Optional[np.ndarray] = None,
) -> dict[str, float]:
    """MCC, F1 и операционный DL для одного порога score.

    Точки без оценки не входят в MCC, F1, precision и recall; из ломаной операционного DL исключаются,
    как отвергнутые.

    Args:
        tau: Кандидат порога (для записи в результат).
        predicted_anomaly: Маска предсказанных anomaly.
        frame: Ряд с time, lat, lon, x, n, y.
        ground_truth_anomaly: Эталонная маска anomaly.
        scored: True в точках с оценкой аномальности; None — оценка есть во всех точках.

    Returns:
        dict[str, float]: tau, mcc, f1, precision, recall, loss_m.
    """
    east_m = frame["x"].to_numpy(dtype=float)
    north_m = frame["n"].to_numpy(dtype=float)
    predicted = np.asarray(predicted_anomaly, dtype=bool)
    truth = np.asarray(ground_truth_anomaly, dtype=bool)
    scored_mask = np.ones(len(predicted), dtype=bool) if scored is None else np.asarray(scored, dtype=bool)
    true_positive, false_positive, false_negative, true_negative = confusion_counts(
        predicted[scored_mask], truth[scored_mask]
    )
    precision_denominator = true_positive + false_positive
    recall_denominator = true_positive + false_negative
    f1_denominator = 2.0 * true_positive + false_positive + false_negative
    precision = true_positive / precision_denominator if precision_denominator > 0.0 else float("nan")
    recall = true_positive / recall_denominator if recall_denominator > 0.0 else float("nan")
    f1 = 2.0 * true_positive / f1_denominator if f1_denominator > 0.0 else float("nan")
    mcc = mcc_from_counts(true_positive, false_positive, false_negative, true_negative)
    loss_m = operational_distance_loss_m(east_m, north_m, truth, predicted | ~scored_mask)
    return {
        "tau": float(tau),
        "mcc": float(mcc),
        "f1": float(f1),
        "precision": float(precision),
        "recall": float(recall),
        "loss_m": float(loss_m),
    }


def _pick_mcc_band_record(band: list[dict[str, float]], tie_key: str) -> dict[str, float]:
    """Среди записей в полосе MCC — минимальный операционный DL.

    Args:
        band: Записи-кандидаты в полосе MCC.
        tie_key: Имя ключа для детерминированного разрешения ничьей.

    Returns:
        dict[str, float]: Запись с минимальным loss_m, затем с максимальными MCC и F1.
    """
    return sorted(band, key=lambda r: (r["loss_m"], -r["mcc"], -r["f1"], r[tie_key]))[0]


def select_threshold_from_records(records: list[dict[str, float]], mcc_band: float) -> dict[str, float]:
    """Выбирает порог по записям кандидатов: max MCC, в полосе mcc_band — min операционный DL.

    Args:
        records: Записи _threshold_record по кандидатам порога.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.

    Raises:
        RuntimeError: Нет допустимых порогов.
    """
    usable = [r for r in records if np.isfinite(r["loss_m"]) and np.isfinite(r["mcc"])]
    if not usable:
        raise RuntimeError("Нет конечного MCC и distance loss для выбора порога.")
    best_mcc = max(r["mcc"] for r in usable)
    band = [r for r in usable if r["mcc"] >= best_mcc - mcc_band - 1e-12]
    chosen = _pick_mcc_band_record(band, "tau")
    return {
        "tau": chosen["tau"],
        "train_mcc": chosen["mcc"],
        "train_f1": chosen["f1"],
        "train_dl_m": chosen["loss_m"],
        "best_mcc": float(best_mcc),
        "band_size": float(len(band)),
    }


def threshold_candidates(scores: np.ndarray, n_quantiles: int = 180, n_log: int = THRESHOLD_LOG_POINTS) -> np.ndarray:
    """Сетка кандидатов порога: квантили score и точки, равномерные в логарифмической шкале.

    Квантили сгущаются там, где много точек; логарифмические точки закрывают провалы распределения
    (score штатного хода и аномалий различаются на порядки). Если score не положителен, вместо
    логарифмической шкалы — равномерная.

    Args:
        scores: Конечные значения score точек с оценкой.
        n_quantiles: Число квантилей на уровнях от 0.01 до 0.995.
        n_log: Число точек между 1-м перцентилем и максимумом score.

    Returns:
        np.ndarray: Отсортированные уникальные кандидаты.
    """
    values = np.asarray(scores, dtype=float)
    quantiles = np.quantile(values, np.linspace(0.01, 0.995, n_quantiles))
    low, high = float(np.quantile(values, 0.01)), float(np.max(values))
    if high <= low:
        return np.unique(quantiles)
    spread = np.geomspace(low, high, n_log) if low > 0.0 else np.linspace(low, high, n_log)
    return np.unique(np.concatenate([quantiles, spread]))


# Порядок и подписи.
TEST_REPORT_METRICS: tuple[tuple[str, str], ...] = (
    ("rmse_m", "RMSE, м"),
    ("max_deviation_m", "Max(d_i), м"),
    ("distance_loss_m", "Distance loss, м"),
    ("delay_median", "Delay median, с"),
    ("delay_mean", "Delay mean, с"),
    ("delay_p95", "Delay p95, с"),
    ("delay_max", "Delay max, с"),
    ("delay_n_on_time", "On-time episodes"),
    ("delay_n_late_or_missed", "Late/missed episodes"),
    ("precision", "Precision"),
    ("recall", "Recall"),
    ("f1", "F1"),
    ("mcc", "MCC"),
    ("unscored_share", "Без оценки, %"),
    ("unscored_anomaly_share", "Из них аномалий, %"),
    ("anomaly_unscored_share", "Аномалий без оценки, %"),
)


def test_report_row(scenario_name: str, metrics: dict[str, float]) -> dict[str, object]:
    """Оставляет только метрики test в фиксированном порядке.

    Args:
        scenario_name: Имя сценария (df1 / df2).
        metrics: Результат evaluate_test / TrajectoryMetrics.compute_all.

    Returns:
        dict[str, object]: Строка сводной таблицы.
    """
    row: dict[str, object] = {"Сценарий": scenario_name}
    for key, _ in TEST_REPORT_METRICS:
        row[key] = metrics.get(key, float("nan"))
    return row


def _format_metric_value(key: str, value) -> str:
    """Форматирует одну метрику test для таблицы отчёта.

    Args:
        key: Имя метрики из TEST_REPORT_METRICS.
        value: Числовое значение или None.

    Returns:
        str: Строка для display (или «—» при пропуске).
    """
    if value is None or (isinstance(value, float) and not np.isfinite(value)):
        return "—"
    if key in {"precision", "recall", "f1", "mcc"}:
        rate = float(value)
        return fmt_num(rate, 4) if abs(rate) >= 1000 else f"{rate:.4f}"
    if key.endswith("_share"):
        return f"{100.0 * float(value):.2f}"
    if key in {"distance_loss_m", "rmse_m", "max_deviation_m"}:
        return fmt_num(float(value), 2)
    if key.startswith("delay_n_"):
        return fmt_num(int(value), 0)
    if key.startswith("delay_"):
        delay = float(value)
        return fmt_num(delay, 2) if abs(delay) >= 1000 else f"{delay:.2f}"
    return str(value)


def display_test_report_table(frame: pd.DataFrame, caption: str = "") -> None:
    """Стилизованная таблица итоговых метрик test (как в Отчёте).

    Args:
        frame: Столбец «Сценарий» и ключи из TEST_REPORT_METRICS.
        caption: Подпись таблицы.
    """
    labels = {key: label for key, label in TEST_REPORT_METRICS}
    renamed = frame.rename(columns=labels)
    for key, label in TEST_REPORT_METRICS:
        if label in renamed.columns:
            renamed[label] = renamed[label].map(lambda v, k=key: _format_metric_value(k, v))
    styled = renamed.style
    if caption:
        styled = styled.set_caption(caption)
    display(styled)


# Модели прогноза (AR, ARIMA): регулярная сетка по времени, оценка по штатному ходу, замена отвергнутых точек.
# Дисперсия диффузного начального состояния, м²: много больше квадрата координат ENU.
_DIFFUSE_VAR_M2 = 1e14


def grid_nodes(time_sec: np.ndarray, step_sec: float) -> tuple[np.ndarray, np.ndarray]:
    """Узлы регулярной сетки по времени и маска точек, оставшихся в своём узле.

    Args:
        time_sec: Время точек по возрастанию, с.
        step_sec: Шаг сетки, с.

    Returns:
        tuple[np.ndarray, np.ndarray]: Номер узла каждой точки и маска первой точки узла
        (остальные точки узла — без оценки).
    """
    node = np.rint((time_sec - time_sec[0]) / step_sec).astype(np.int64)
    first = np.ones(len(node), dtype=bool)
    first[1:] = node[1:] != node[:-1]
    return node, first


def _forecast_model(series: np.ndarray, order: tuple[int, int, int], trend: str) -> ARIMA:
    """Модель statsmodels с ограничениями стационарности и обратимости."""
    return ARIMA(series, order=order, trend=trend, enforce_stationarity=True, enforce_invertibility=True)


# Штраф целевой функции за параметры на границе стационарности или обратимости.
_INADMISSIBLE_PENALTY = 1e300


def _forecast_params_admissible(model: ARIMA, params: np.ndarray) -> bool:
    """Корни AR- и MA-многочленов строго внутри единичного круга: стационарность и обратимость."""
    for prefix, sign in (("ar.", -1.0), ("ma.", 1.0)):
        coefs = np.array([value for name, value in zip(model.param_names, params) if name.startswith(prefix)])
        if coefs.size and np.any(np.abs(np.roots(np.r_[1.0, sign * coefs])) >= 1.0 - 1e-9):
            return False
    return True


def _start_candidates(model: ARIMA, order: tuple[int, int, int]) -> list[np.ndarray]:
    """Начальные параметры оптимизации: из statsmodels и с AR-коэффициентами по МНК.

    Вблизи единичного корня оптимизация из одного старта попадает в локальный максимум; оба старта сжаты
    внутрь области стационарности и обратимости.

    Args:
        model: Модель statsmodels самого длинного отрезка.
        order: Порядок (p, d, q).

    Returns:
        list[np.ndarray]: Допустимые наборы параметров.
    """
    base = np.asarray(model.start_params, dtype=float)
    candidates = [base.copy()]
    target = np.diff(model.endog.ravel()[~np.isnan(model.endog.ravel())], n=order[1])
    p_order = order[0]
    if p_order and len(target) > 2 * p_order:
        lags = np.column_stack([target[p_order - lag : len(target) - lag] for lag in range(1, p_order + 1)])
        ols = np.linalg.lstsq(lags, target[p_order:], rcond=None)[0]
        alternative = base.copy()
        alternative[[name.startswith("ar.") for name in model.param_names]] = ols
        candidates.append(alternative)
    lagged = np.array([name.startswith(("ar.", "ma.")) for name in model.param_names])
    for start in candidates:
        while not _forecast_params_admissible(model, start):
            start[lagged] *= 0.9
    return candidates


def fit_forecast_axis(
    node: np.ndarray, values: np.ndarray, starts: np.ndarray, order: tuple[int, int, int], trend: str
) -> np.ndarray:
    """Параметры ARIMA одной оси: максимум суммы логарифмов правдоподобия отрезков с общими параметрами.

    Отрезки — обучающие отрезки штатного хода (move_segments); внутри отрезка пустые узлы — NaN.

    Args:
        node: Узлы наблюдений по возрастанию (точки штатного хода).
        values: Координата наблюдений, м.
        starts: Маска первой точки отрезка.
        order: Порядок (p, d, q).
        trend: Член тренда statsmodels.

    Returns:
        np.ndarray: Параметры в порядке param_names statsmodels.

    Raises:
        RuntimeError: Нет отрезка достаточной длины.
    """
    min_points = order[0] + order[1] + 2
    cuts = np.flatnonzero(starts)[1:]
    models: list[ARIMA] = []
    for index in np.split(np.arange(len(node)), cuts):
        if len(index) < min_points:
            continue
        series = np.full(node[index[-1]] - node[index[0]] + 1, np.nan)
        # Уровень отрезка — его первая точка: координата отсчитывается от неё.
        series[node[index] - node[index[0]]] = values[index] - values[index[0]]
        model = _forecast_model(series, order, trend)
        # Условное правдоподобие: диффузное начальное состояние, первые p + d наблюдений отрезка не входят —
        # как точки без оценки в детекторе.
        model.ssm.initialize_approximate_diffuse(_DIFFUSE_VAR_M2)
        model.ssm.loglikelihood_burn = order[0] + order[1]
        models.append(model)
    if not models:
        raise RuntimeError("Нет отрезка штатного хода для оценки параметров.")
    longest = max(models, key=lambda model: model.nobs)

    def negative_loglike(unconstrained: np.ndarray) -> float:
        """Минус сумма логарифмов правдоподобия отрезков; вне допустимой области — штраф."""
        params = longest.transform_params(unconstrained)
        if not _forecast_params_admissible(longest, params):
            return _INADMISSIBLE_PENALTY
        total = sum(model.loglike(params) for model in models)
        return -total if np.isfinite(total) else _INADMISSIBLE_PENALTY

    results = [
        minimize(negative_loglike, longest.untransform_params(start), method="L-BFGS-B")
        for start in _start_candidates(longest, order)
    ]
    return longest.transform_params(min(results, key=lambda result: result.fun).x)


def forecast_matrices(order: tuple[int, int, int], trend: str, params: np.ndarray) -> dict[str, np.ndarray]:
    """Матрицы формы пространства состояний statsmodels для заданных параметров.

    Args:
        order: Порядок (p, d, q).
        trend: Член тренда statsmodels.
        params: Параметры модели.

    Returns:
        dict[str, np.ndarray]: design (k,), transition (k, k), noise (k, k) и кэш горизонтов.
    """
    model = _forecast_model(np.zeros(16), order, trend)
    model.update(params)
    k_states = model.k_states
    selection = np.asarray(model.ssm["selection"], dtype=float).reshape(k_states, -1)
    state_cov = np.asarray(model.ssm["state_cov"], dtype=float).reshape(selection.shape[1], -1)
    return {
        "design": np.asarray(model.ssm["design"], dtype=float).reshape(k_states),
        "transition": np.asarray(model.ssm["transition"], dtype=float).reshape(k_states, k_states),
        "noise": selection @ state_cov @ selection.T,
        "horizons": {},
    }


def _horizon_matrices(axis: dict[str, np.ndarray], horizon: int) -> tuple[np.ndarray, np.ndarray]:
    """Переход на horizon шагов и накопленный шум: T^h и сумма T^m Q T^m' по m < h (с кэшем)."""
    cache = axis["horizons"]
    if horizon not in cache:
        power = np.eye(len(axis["design"]))
        noise = np.zeros_like(power)
        for _ in range(horizon):
            noise = axis["transition"] @ noise @ axis["transition"].T + axis["noise"]
            power = axis["transition"] @ power
        cache[horizon] = (power, noise)
    return cache[horizon]


def fit_forecast_detector(train: pd.DataFrame, order: tuple[int, int, int], trend: str) -> dict[str, object]:
    """Оценивает модель прогноза по точкам штатного хода обучающей выборки, по двум осям ENU.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.
        order: Порядок (p, d, q).
        trend: Член тренда statsmodels.

    Returns:
        dict[str, object]: params и axes (по осям x, n), n_unscored — число точек без оценки в начале отрезка,
        speed_limit — предел скорости перезапуска, м/с.
    """
    xy = enu_xy_from_frame(train)
    node, first = grid_nodes(time_seconds_from_frame(train), GRID_STEP_SEC)
    index, starts = segments_layout(move_segments(train))
    # Из точек одного узла остаётся первая; отрезок, потерявший первую точку, начинается со следующей.
    keep = first[index]
    segment = np.cumsum(starts)[keep]
    starts = np.r_[True, segment[1:] != segment[:-1]]
    index = index[keep]
    params = [fit_forecast_axis(node[index], xy[index, column], starts, order, trend) for column in range(2)]
    return {
        "params": params,
        "axes": [forecast_matrices(order, trend, value) for value in params],
        "n_unscored": order[0] + order[1],
        "speed_limit": restart_speed_limit(train),
    }


# Канонический кандидат порога NIS моделей прогноза.
FORECAST_CHI999 = float(chi2.ppf(0.999, 2))


def forecast_detect(
    frame: pd.DataFrame, detector: dict[str, object], taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """Причинный прогноз на сетке с заменой отвергнутых точек, сразу для нескольких порогов.

    Оценка — NIS: сумма по осям квадрата ошибки прогноза, делённого на дисперсию прогноза. Отвергнутая точка
    (оценка больше порога) не обновляет состояние. Прогноз начинается заново после разрыва по времени и после
    серии отвергнутых точек длиннее разрыва, если точка достижима от последней принятой (restart_allowed);
    первые n_unscored точек отрезка — без оценки.

    Args:
        frame: DataFrame с колонками x, n и time.
        detector: Результат fit_forecast_detector.
        taus: Пороги; np.inf — без замены.

    Returns:
        tuple[np.ndarray, np.ndarray]: NIS shape (C, N), NaN — точка без оценки;
        маска отвергнутых точек shape (C, N).
    """
    xy = enu_xy_from_frame(frame)
    time_sec = time_seconds_from_frame(frame)
    node, first = grid_nodes(time_sec, GRID_STEP_SEC)
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    n_cand, count = len(thresholds), len(xy)
    axes = detector["axes"]
    k_states = len(axes[0]["design"])
    scores = np.full((n_cand, count), np.nan, dtype=np.float32)
    rejected = np.zeros((n_cand, count), dtype=bool)
    state = np.zeros((2, n_cand, k_states))
    cov = np.zeros((2, n_cand, k_states, k_states))
    accepted_since = np.zeros(n_cand, dtype=np.int64)
    run_start = np.full(n_cand, np.nan)
    # Последняя принятая точка: от неё проверяется достижимость точки перезапуска.
    last_xy = np.zeros((n_cand, 2))
    last_time = np.zeros(n_cand)
    design = np.stack([axis["design"] for axis in axes])
    # Уровень модели — первая точка отрезка; задаётся при перезапуске прогноза.
    level = np.zeros((2, n_cand))
    prev_node, prev_time = None, None
    for index in np.flatnonzero(first):
        moment = time_sec[index]
        if prev_node is None or moment - prev_time > TIME_GAP_SEC:
            restart = np.ones(n_cand, dtype=bool)
        else:
            restart = restart_allowed(xy[index], moment, last_xy, last_time, run_start, detector["speed_limit"])
            for axis_index, axis in enumerate(axes):
                power, noise = _horizon_matrices(axis, int(node[index] - prev_node))
                state[axis_index] = state[axis_index] @ power.T
                cov[axis_index] = power @ cov[axis_index] @ power.T + noise
        if np.any(restart):
            state[:, restart] = 0.0
            cov[:, restart] = np.eye(k_states) * _DIFFUSE_VAR_M2
            accepted_since[restart] = 0
            level[:, restart] = xy[index][:, None]
            run_start[restart] = np.nan
        innovation = xy[index][:, None] - np.einsum("ack,ak->ac", state, design) - level
        cov_design = np.einsum("ackl,al->ack", cov, design)
        variance = np.einsum("ack,ak->ac", cov_design, design)
        nis = np.sum(innovation**2 / variance, axis=0)
        unscored = accepted_since < detector["n_unscored"]
        reject = ~unscored & (nis > thresholds)
        scores[:, index] = np.where(unscored, np.nan, nis)
        rejected[:, index] = reject
        accept = ~reject
        gain = cov_design / variance[:, :, None]
        state[:, accept] += gain[:, accept] * innovation[:, accept, None]
        cov[:, accept] -= gain[:, accept, :, None] * cov_design[:, accept, None, :]
        accepted_since[accept] += 1
        last_xy[accept] = xy[index]
        last_time[accept] = moment
        run_start = np.where(reject, np.where(np.isnan(run_start), moment, run_start), np.nan)
        prev_node, prev_time = node[index], moment
    return scores, rejected


def select_forecast_threshold(
    frame: pd.DataFrame, detector: dict[str, object], mcc_band: float, n_quantiles: int = 180
) -> dict[str, float]:
    """Подбирает порог модели прогноза: сетка по оценке без замены и χ²₀.₉₉₉(2), прогон с заменой на каждого кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, n, y).
        detector: Результат fit_forecast_detector.
        mcc_band: Допуск по MCC относительно лучшего значения.
        n_quantiles: Число квантилей в сетке порогов (см. threshold_candidates).

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    base_scores, _ = forecast_detect(frame, detector, np.array([np.inf]))
    finite = base_scores[0][np.isfinite(base_scores[0])]
    candidates = np.unique(np.r_[threshold_candidates(finite, n_quantiles), FORECAST_CHI999])
    scores, rejected = forecast_detect(frame, detector, candidates)
    ground_truth_anomaly = frame["y"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


def forecast_predict(
    frame: pd.DataFrame, detector: dict[str, object], tau: float
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Разметка модели прогноза при одном пороге.

    Args:
        frame: DataFrame с колонками x, n и time.
        detector: Результат fit_forecast_detector.
        tau: Порог NIS.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: Оценка аномальности (NaN — без оценки), маска anomaly,
        маска точек с оценкой.
    """
    scores, rejected = forecast_detect(frame, detector, np.array([tau]))
    values = scores[0].astype(np.float64)
    return values, rejected[0], ~np.isnan(values)


def time_segments(frame: pd.DataFrame, mask: np.ndarray) -> list[np.ndarray]:
    """Индексы точек маски, разрезанные по разрывам по времени; из точек с одинаковым временем — первая.

    Args:
        frame: DataFrame с колонкой time.
        mask: Отбираемые точки.

    Returns:
        list[np.ndarray]: Индексы точек каждого отрезка.
    """
    time_sec = time_seconds_from_frame(frame)
    index = np.flatnonzero(mask)
    if index.size == 0:
        return []
    index = index[np.r_[True, np.diff(time_sec[index]) > 0.0]]
    return np.split(index, np.flatnonzero(np.diff(time_sec[index]) > TIME_GAP_SEC) + 1)


def move_segments(frame: pd.DataFrame) -> list[np.ndarray]:
    """Обучающие отрезки штатного хода: подряд идущие точки штатного хода между разрывами по времени.

    Отрезок рвётся на каждой аномальной точке: точки штатного хода по разные стороны серии аномалий
    не склеиваются, и прыжок через серию не попадает в обучение как обычный переход.

    Args:
        frame: DataFrame с колонками time и y.

    Returns:
        list[np.ndarray]: Индексы точек каждого отрезка.
    """
    move = frame["y"].astype(str).to_numpy() == "move"
    segments = []
    for segment in time_segments(frame, np.ones(len(frame), dtype=bool)):
        cuts = np.flatnonzero(np.diff(move[segment].astype(np.int8)) != 0) + 1
        segments.extend(part for part in np.split(segment, cuts) if move[part[0]])
    return segments


def segments_layout(segments: list[np.ndarray]) -> tuple[np.ndarray, np.ndarray]:
    """Точки отрезков подряд и маска первой точки отрезка.

    Args:
        segments: Индексы точек каждого отрезка.

    Returns:
        tuple[np.ndarray, np.ndarray]: индексы точек отрезков подряд (P,); маска первой точки отрезка (P,).
    """
    if not segments:
        raise RuntimeError("Нет отрезков для признаков.")
    first = np.zeros(sum(len(segment) for segment in segments), dtype=bool)
    first[np.r_[0, np.cumsum([len(segment) for segment in segments])[:-1]]] = True
    return np.concatenate(segments), first


def move_speeds(train: pd.DataFrame) -> np.ndarray:
    """Скорости между соседними точками обучающих отрезков штатного хода, м/с.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.

    Returns:
        np.ndarray: Скорости переходов всех отрезков.
    """
    xy = enu_xy_from_frame(train)
    time_sec = time_seconds_from_frame(train)
    speeds = []
    for segment in move_segments(train):
        if len(segment) > 1:
            step = np.diff(xy[segment], axis=0)
            speeds.append(np.hypot(step[:, 0], step[:, 1]) / np.diff(time_sec[segment]))
    if not speeds:
        raise RuntimeError("Нет переходов штатного хода в обучающей выборке.")
    return np.concatenate(speeds)


def restart_speed_limit(train: pd.DataFrame) -> float:
    """Предел скорости перезапуска прогноза — наибольшая скорость штатного хода обучающей выборки, м/с.

    Общий для всех моделей прогноза (AR, ARIMA, Kalman, GP, GP-EKF): одинаковые условия перезапуска.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.

    Returns:
        float: Предел скорости, м/с.
    """
    return float(np.max(move_speeds(train)))


def restart_allowed(
    point: np.ndarray,
    moment: float,
    last_xy: np.ndarray,
    last_time: np.ndarray,
    run_start: np.ndarray,
    speed_limit: float,
) -> np.ndarray:
    """Перезапуск прогноза по серии отвергнутых точек, для нескольких порогов сразу.

    Серия отвергнутых точек длиннее разрыва по времени перезапускает прогноз, только если точка достижима от
    последней принятой со скоростью не выше предела: иначе опорной стала бы точка смещённого трека аномалии.

    Args:
        point: Координаты ENU точки (2,), м.
        moment: Время точки, с.
        last_xy: Последние принятые точки (C, 2), м.
        last_time: Их время (C,), с.
        run_start: Начало серии отвергнутых точек (C,), с; NaN — серии нет.
        speed_limit: Предел скорости, м/с (restart_speed_limit).

    Returns:
        np.ndarray: Маска перезапуска (C,).
    """
    step = point[None, :] - last_xy
    speed = np.hypot(step[:, 0], step[:, 1]) / np.maximum(moment - last_time, 1e-9)
    return (moment - run_start > TIME_GAP_SEC) & (speed <= speed_limit)


def feature_keys(before: np.ndarray, rows: np.ndarray) -> np.ndarray:
    """Ключи признаков точек во втором проходе: точка, её предшественник и предшественник предшественника.

    Признаки точки (dbscan_features) определяются этой тройкой: равные ключи — равные признаки.

    Args:
        before: Позиция предыдущей принятой точки для каждой точки; -1 — нет.
        rows: Позиции точек.

    Returns:
        np.ndarray: Ключи (R, 3).
    """
    prev = before[rows]
    return np.column_stack([rows, prev, np.where(prev >= 0, before[np.maximum(prev, 0)], -1)])


def cached_rows(
    keys: np.ndarray, cache: dict[bytes, object], compute: Callable[[np.ndarray], np.ndarray]
) -> np.ndarray:
    """Результаты по ключам: найденные — из кэша, недостающие — compute и в кэш.

    Args:
        keys: Ключи (R, d), целые; в одном вызове не повторяются.
        cache: Кэш ключ → результат строки, общий для кандидатов порога.
        compute: Результаты недостающих строк по маске (R,) — массив (M,) или (M, ...).

    Returns:
        np.ndarray: Результаты (R,) или (R, ...).
    """
    packed = np.ascontiguousarray(keys, dtype=np.int64)
    packed = packed.view(np.dtype((np.void, 8 * packed.shape[1]))).ravel().tolist()
    missing = np.array([key not in cache for key in packed], dtype=bool)
    if np.any(missing):
        values = compute(missing)
        cache.update(zip((key for key, absent in zip(packed, missing) if absent), values))
    return np.array([cache[key] for key in packed])


def dbscan_segments(frame: pd.DataFrame, mask: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Точки маски по отрезкам между разрывами по времени и маска первой точки отрезка.

    Args:
        frame: DataFrame с колонкой time.
        mask: Отбираемые точки.

    Returns:
        tuple[np.ndarray, np.ndarray]: индексы точек отрезков подряд (P,); маска первой точки отрезка (P,).
    """
    return segments_layout(time_segments(frame, mask))


def dbscan_features(xy: np.ndarray, time_sec: np.ndarray, before: np.ndarray) -> np.ndarray:
    """Признаки (v, a_long, a_lat) по предыдущей принятой точке ``before`` и её предыдущей принятой точке.

    Args:
        xy: Координаты ENU точек отрезков подряд (P, 2), м.
        time_sec: Время точек, с.
        before: Позиция предыдущей принятой точки того же отрезка; -1 — нет.

    Returns:
        np.ndarray: (P, 3): скорость, м/с; продольное и поперечное ускорение, м/с²; NaN — точка без оценки.
    """
    features = np.full((len(xy), 3), np.nan)
    prev = before
    prev2 = np.where(prev >= 0, before[np.maximum(prev, 0)], -1)
    valid = (prev >= 0) & (prev2 >= 0)
    cur, one, two = np.flatnonzero(valid), prev[valid], prev2[valid]
    step = xy[cur] - xy[one]
    back = xy[one] - xy[two]
    dt = time_sec[cur] - time_sec[one]
    speed = np.hypot(step[:, 0], step[:, 1]) / dt
    prev_speed = np.hypot(back[:, 0], back[:, 1]) / (time_sec[one] - time_sec[two])
    turn = wrap_angle(np.arctan2(step[:, 0], step[:, 1]) - np.arctan2(back[:, 0], back[:, 1]))
    features[cur] = np.column_stack([speed, (speed - prev_speed) / dt, speed * turn / dt])
    return features


def rpca_window_starts(first: np.ndarray, scored: np.ndarray, length: int) -> np.ndarray:
    """Начала окон из ``length`` подряд идущих точек с признаками внутри одного отрезка.

    Args:
        first: Маска первой точки отрезка (P,).
        scored: Маска точек с признаками (P,).
        length: Длина окна, точки.

    Returns:
        np.ndarray: Позиции начала окон.
    """
    count = len(first)
    if count < length:
        return np.empty(0, dtype=np.int64)
    segment = np.cumsum(first)
    scored_sum = np.r_[0, np.cumsum(scored)]
    starts = np.arange(count - length + 1)
    inside = scored_sum[starts + length] - scored_sum[starts] == length
    return starts[inside & (segment[starts + length - 1] == segment[starts])]


def train_with_early_stopping(model: nn.Module, loader: DataLoader, batch_loss, calib_loss) -> dict[str, object]:
    """Adam до NN_MAX_EPOCHS эпох с ранней остановкой и снижением скорости обучения на плато.

    Эпоха лучше прежней, если ошибка на калибровочном блоке упала больше чем на NN_MIN_DELTA (относительно); после
    NN_PATIENCE эпох без улучшения обучение прекращается, веса — лучшей эпохи. Скорость обучения уменьшается
    в NN_LR_FACTOR раз после NN_LR_PATIENCE эпох без улучшения.

    Args:
        model: Сеть на DEVICE.
        loader: Пакеты обучающих тензоров.
        batch_loss: Функция (сеть, список тензоров пакета на DEVICE) -> скалярная ошибка для градиента.
        calib_loss: Функция (сеть) -> ошибка на калибровочном блоке (float).

    Returns:
        dict[str, object]: epochs, calib_loss — лучшая ошибка калибровки, stopped_by_cap — остановка по лимиту эпох.
    """
    optimizer = torch.optim.Adam(model.parameters(), lr=TORCH_LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, factor=NN_LR_FACTOR, patience=NN_LR_PATIENCE, threshold=NN_MIN_DELTA
    )
    best_loss, best_state, stale, epochs = np.inf, None, 0, 0
    for epoch in range(1, NN_MAX_EPOCHS + 1):
        model.train()
        for batch in loader:
            optimizer.zero_grad()
            loss = batch_loss(model, [tensor.to(DEVICE) for tensor in batch])
            loss.backward()
            optimizer.step()
        model.eval()
        with torch.no_grad():
            value = float(calib_loss(model))
        scheduler.step(value)
        epochs = epoch
        if best_state is None or value < best_loss - NN_MIN_DELTA * abs(best_loss):
            best_loss, stale = value, 0
            best_state = {key: tensor.detach().clone() for key, tensor in model.state_dict().items()}
        else:
            stale += 1
            if stale >= NN_PATIENCE:
                break
    model.load_state_dict(best_state)
    model.eval()
    stopped_by_cap = stale < NN_PATIENCE
    if stopped_by_cap:
        print(f"  предупреждение: обучение упёрлось в лимит {NN_MAX_EPOCHS} эпох, плато не достигнуто")
    return {"epochs": epochs, "calib_loss": best_loss, "stopped_by_cap": stopped_by_cap}

## Модели

Представители семейств — по самой быстрой модели: модели семейства учатся на одних и тех же данных и одинаково от них зависят.

| Семейство | Модели | Представитель |
| --- | --- | --- |
| прогноз | AR, ARIMA, фильтр Калмана, гауссовский процесс, GP-EKF | фильтр Калмана |
| новизна по признакам | Isolation Forest, One-Class SVM, Robust PCA | Isolation Forest |
| плотность | DBSCAN, LOF | DBSCAN |
| оба класса | HMM | HMM |
| без обучения параметров | MPCT, фильтр Хэмпела | MPCT |
| окна, нейросети | LSTM-автоэнкодер, TCN, DONUT | проверяется требованием к числу окон |

Теория — в example_model.ipynb.

In [ ]:
MPCT_GRID_SIZE = 30
MPCT_GRID_LOW_PERCENTILE = 95.0
MPCT_GRID_HIGH_FACTOR = 3.0


def _mpct_segments(frame: pd.DataFrame) -> tuple[np.ndarray, list[np.ndarray]]:
    """Точки ряда по отрезкам между разрывами по времени (из точек с одинаковым временем — первая)."""
    segments = time_segments(frame, np.ones(len(frame), dtype=bool))
    return np.concatenate(segments), segments


def mpct_select_points(xy: np.ndarray, time_sec: np.ndarray, v_max: np.ndarray) -> np.ndarray:
    """Самая длинная допустимая подпоследовательность одного отрезка для нескольких v_+ (динамическое программирование).

    Соседние выбранные точки допустимы, если между ними больше TIME_GAP_SEC (цепочка заново) или скорость в плоскости
    ENU не выше v_+. При равной длине — ближайший по времени предшественник и самый поздний конец.

    Args:
        xy: Координаты ENU точек отрезка (P, 2), м.
        time_sec: Время точек, с, строго по возрастанию.
        v_max: Пределы скорости (C,), м/с.

    Returns:
        np.ndarray: Маска выбранных точек (C, P).
    """
    count, n_cand = len(xy), len(v_max)
    best = np.zeros((n_cand, count), dtype=np.int64)
    pred = np.full((n_cand, count), -1, dtype=np.int64)
    run_best = np.zeros(n_cand, dtype=np.int64)
    run_arg = np.full(n_cand, -1, dtype=np.int64)
    rows = np.arange(n_cand)
    oldest = 0
    for index in range(count):
        # Точки старше TIME_GAP_SEC — предшественники при перезапуске цепочки: накопленный максимум.
        while oldest < index and time_sec[index] - time_sec[oldest] > TIME_GAP_SEC:
            better = best[:, oldest] >= run_best
            run_best = np.where(better, best[:, oldest], run_best)
            run_arg = np.where(better, oldest, run_arg)
            oldest += 1
        length, source = run_best + 1, run_arg.copy()
        if oldest < index:
            step = xy[index] - xy[oldest:index]
            speed = np.hypot(step[:, 0], step[:, 1]) / (time_sec[index] - time_sec[oldest:index])
            window = np.where(speed[None, :] <= v_max[:, None], best[:, oldest:index], 0)
            # Последний максимум — ближайший по времени предшественник.
            last = window.shape[1] - 1 - np.argmax(window[:, ::-1], axis=1)
            top = window[rows, last]
            use = (top > 0) & (top + 1 >= length)
            length = np.where(use, top + 1, length)
            source = np.where(use, oldest + last, source)
        best[:, index] = length
        pred[:, index] = source
    chosen = np.zeros((n_cand, count), dtype=bool)
    for row in range(n_cand):
        cursor = count - 1 - int(np.argmax(best[row, ::-1]))
        while cursor >= 0:
            chosen[row, cursor] = True
            cursor = pred[row, cursor]
    return chosen


def mpct_detect(frame: pd.DataFrame, v_max: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Разметка MPCT для нескольких v_+: аномалия — точка вне самой длинной допустимой подпоследовательности отрезка.

    Args:
        frame: DataFrame с колонками x, n и time.
        v_max: Пределы скорости (C,), м/с.

    Returns:
        tuple[np.ndarray, np.ndarray]: маска аномалий (C, N); маска точек с решением (N,) — без повторов времени.
    """
    limits = np.atleast_1d(np.asarray(v_max, dtype=float))
    xy = enu_xy_from_frame(frame)
    time_sec = time_seconds_from_frame(frame)
    index, segments = _mpct_segments(frame)
    rejected = np.zeros((len(limits), len(frame)), dtype=bool)
    for segment in segments:
        rejected[:, segment] = ~mpct_select_points(xy[segment], time_sec[segment], limits)
    scored = np.zeros(len(frame), dtype=bool)
    scored[index] = True
    return rejected, scored


def mpct_speed_grid(train: pd.DataFrame) -> np.ndarray:
    """Сетка v_+: от 95-го процентиля скоростей между соседними точками штатного хода до трёх их максимумов.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.

    Returns:
        np.ndarray: MPCT_GRID_SIZE значений, м/с.
    """
    speeds = move_speeds(train)
    low = np.percentile(speeds, MPCT_GRID_LOW_PERCENTILE)
    return np.geomspace(low, MPCT_GRID_HIGH_FACTOR * speeds.max(), MPCT_GRID_SIZE)


def select_mpct_speed(frame: pd.DataFrame, mcc_band: float) -> dict[str, float]:
    """Подбирает v_+ по сетке: максимум MCC, в полосе — минимальная операционная ошибка длины.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, n, y).
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau (выбранное v_+), train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    grid = mpct_speed_grid(frame)
    rejected, scored = mpct_detect(frame, grid)
    ground_truth_anomaly = frame["y"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(v_max), rejected[row], frame, ground_truth_anomaly, scored)
        for row, v_max in enumerate(grid)
    ]
    return select_threshold_from_records(records, mcc_band)


In [ ]:
KALMAN_SIGMA_ACC_START = 0.35
KALMAN_SIGMA_MEAS_START = 40.0
KALMAN_SIGMA_ACC_MIN = 1e-3
KALMAN_SIGMA_MEAS_MIN = 0.5
KALMAN_CHI999 = float(chi2.ppf(0.999, 2))


def kalman_process_noise(dt: np.ndarray, sigma_acc: float) -> np.ndarray:
    """Ковариация шума процесса одной оси при дискретном белом ускорении, shape (2, 2, n).

    Args:
        dt: Шаги времени, с.
        sigma_acc: СКО ускорения, м/с².

    Returns:
        np.ndarray: Матрицы sigma_acc² · [[dt⁴/4, dt³/2], [dt³/2, dt²]].
    """
    dt = np.asarray(dt, dtype=float)
    noise = np.array([[dt**4 / 4.0, dt**3 / 2.0], [dt**3 / 2.0, dt**2]])
    return noise * sigma_acc**2


def kalman_likelihood_data(train: pd.DataFrame) -> dict[str, np.ndarray]:
    """Точки штатного хода обучающей выборки для условного правдоподобия.

    Отрезки — обучающие отрезки штатного хода (move_segments); отрезки короче трёх точек отбрасываются.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.

    Returns:
        dict[str, np.ndarray]: xy (n, 2), step — интервал до следующей точки, с; boundary — последняя точка
        отрезка; counted — точки, входящие в правдоподобие (с третьей точки отрезка).
    """
    segments = [segment for segment in move_segments(train) if len(segment) >= 3]
    if not segments:
        raise RuntimeError("Нет отрезка штатного хода из трёх точек для оценки фильтра Калмана.")
    index = np.concatenate(segments)
    time_sec = time_seconds_from_frame(train)[index]
    lengths = np.array([len(segment) for segment in segments])
    ends = np.cumsum(lengths) - 1
    boundary = np.zeros(len(index), dtype=bool)
    boundary[ends] = True
    counted = np.ones(len(index), dtype=bool)
    counted[ends - lengths + 1] = False
    counted[ends - lengths + 2] = False
    return {
        "xy": enu_xy_from_frame(train)[index],
        "step": np.r_[np.diff(time_sec), 0.0],
        "boundary": boundary,
        "counted": counted,
    }


def kalman_loglike(data: dict[str, np.ndarray], sigma_acc: float, sigma_meas: float) -> float:
    """Условное логарифмическое правдоподобие модели постоянной скорости по двум осям (statsmodels).

    На границе отрезков состояние сбрасывается в диффузное; первые две точки отрезка не входят.

    Args:
        data: Результат kalman_likelihood_data.
        sigma_acc: СКО ускорения, м/с².
        sigma_meas: СКО измерения, м.

    Returns:
        float: Сумма логарифмов правдоподобия.
    """
    step = data["step"]
    transition = np.zeros((2, 2, len(step)))
    transition[0, 0] = transition[1, 1] = 1.0
    transition[0, 1] = step
    noise = kalman_process_noise(step, sigma_acc)
    transition[:, :, data["boundary"]] = 0.0
    noise[:, :, data["boundary"]] = np.eye(2)[:, :, None] * _DIFFUSE_VAR_M2
    total = 0.0
    for column in range(2):
        values = data["xy"][:, column]
        kalman = KalmanFilter(k_endog=1, k_states=2, k_posdef=2)
        kalman.bind(values - values[0])
        kalman["design"] = np.array([[1.0, 0.0]])
        kalman["obs_cov"] = np.array([[sigma_meas**2]])
        kalman["transition"] = transition
        kalman["selection"] = np.eye(2)
        kalman["state_cov"] = noise
        kalman.initialize_approximate_diffuse(_DIFFUSE_VAR_M2)
        total += float(np.sum(kalman.loglikeobs()[data["counted"]]))
    return total


def fit_kalman_on_train(train: pd.DataFrame) -> tuple[float, float]:
    """Оценивает sigma_acc и sigma_meas по максимуму условного правдоподобия на штатном ходу.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.

    Returns:
        tuple[float, float]: (sigma_acc, м/с²; sigma_meas, м), снизу ограничены KALMAN_SIGMA_ACC_MIN
        и KALMAN_SIGMA_MEAS_MIN.
    """
    data = kalman_likelihood_data(train)
    # Нижние границы: при меньших sigma правдоподобие statsmodels численно вырождается в константу.
    lower = np.log([KALMAN_SIGMA_ACC_MIN, KALMAN_SIGMA_MEAS_MIN])
    result = minimize(
        lambda log_sigma: -kalman_loglike(data, *np.exp(log_sigma)),
        np.log([KALMAN_SIGMA_ACC_START, KALMAN_SIGMA_MEAS_START]),
        method="L-BFGS-B",
        bounds=[(lower[0], None), (lower[1], None)],
    )
    if np.any(np.isclose(result.x, lower)):
        print("  предупреждение: оценка σ фильтра Калмана на нижней границе")
    sigma_acc, sigma_meas = np.exp(result.x)
    return float(sigma_acc), float(sigma_meas)


def kalman_detect(
    frame: pd.DataFrame, sigma_acc: float, sigma_meas: float, speed_limit: float, taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """NIS фильтра постоянной скорости с заменой отвергнутых точек, сразу для нескольких порогов.

    Отвергнутая точка состояние не обновляет. Прогноз начинается заново после разрыва по времени и после
    серии отвергнутых точек длиннее разрыва, если точка достижима от последней принятой (restart_allowed);
    начальное состояние — по первым двум точкам отрезка, они без оценки. Из точек с одинаковым временем остаётся первая, остальные — без оценки.

    Args:
        frame: DataFrame с колонками x, n и time.
        sigma_acc: СКО ускорения, м/с².
        sigma_meas: СКО измерения, м.
        speed_limit: Предел скорости перезапуска, м/с (restart_speed_limit).
        taus: Пороги NIS; np.inf — без замены.

    Returns:
        tuple[np.ndarray, np.ndarray]: NIS shape (C, N), NaN — точка без оценки; маска отвергнутых точек.
    """
    xy = enu_xy_from_frame(frame)
    time_sec = time_seconds_from_frame(frame)
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    n_cand, count = len(thresholds), len(xy)
    meas_cov = np.eye(2) * sigma_meas**2
    scores = np.full((n_cand, count), np.nan, dtype=np.float32)
    rejected = np.zeros((n_cand, count), dtype=bool)
    state = np.zeros((n_cand, 4))
    cov = np.zeros((n_cand, 4, 4))
    first_point = np.zeros((n_cand, 2))
    first_time = np.zeros(n_cand)
    phase = np.zeros(n_cand, dtype=np.int64)
    run_start = np.full(n_cand, np.nan)
    # Последняя принятая точка: от неё проверяется достижимость точки перезапуска.
    last_xy = np.zeros((n_cand, 2))
    last_time = np.zeros(n_cand)
    transition = np.eye(4)
    prev_time = None
    for index in range(count):
        moment = time_sec[index]
        if prev_time is not None and moment - prev_time <= 0.0:
            continue
        point = xy[index]
        if prev_time is None or moment - prev_time > TIME_GAP_SEC:
            restart = np.ones(n_cand, dtype=bool)
        else:
            restart = restart_allowed(point, moment, last_xy, last_time, run_start, speed_limit)
            dt = moment - prev_time
            transition[0, 2] = transition[1, 3] = dt
            axis_noise = kalman_process_noise(np.array([dt]), sigma_acc)[:, :, 0]
            process = np.zeros((4, 4))
            process[np.ix_([0, 2], [0, 2])] = axis_noise
            process[np.ix_([1, 3], [1, 3])] = axis_noise
            state = state @ transition.T
            cov = transition @ cov @ transition.T + process
        phase[restart] = 0
        run_start[restart] = np.nan
        start, second, active = phase == 0, phase == 1, phase >= 2
        first_point[start] = point
        first_time[start] = moment
        if np.any(second):
            pair_dt = moment - first_time[second]
            state[second, :2] = point
            state[second, 2:] = (point - first_point[second]) / pair_dt[:, None]
            cov[second] = 0.0
            cov[second, :2, :2] = meas_cov
            cov[second, :2, 2:] = cov[second, 2:, :2] = meas_cov / pair_dt[:, None, None]
            cov[second, 2:, 2:] = 2.0 * meas_cov / pair_dt[:, None, None] ** 2
        innovation = point - state[:, :2]
        innovation_cov = cov[:, :2, :2] + meas_cov
        solved = np.linalg.solve(innovation_cov, innovation[:, :, None])[:, :, 0]
        nis = np.einsum("ck,ck->c", innovation, solved)
        reject = active & (nis > thresholds)
        scores[:, index] = np.where(active, nis, np.nan)
        rejected[:, index] = reject
        update = active & ~reject
        gain = cov[update][:, :, :2] @ np.linalg.inv(innovation_cov[update])
        state[update] += np.einsum("ckl,cl->ck", gain, innovation[update])
        cov[update] -= gain @ cov[update][:, :2, :]
        phase[start | second] += 1
        last_xy[~reject] = point
        last_time[~reject] = moment
        run_start = np.where(
            reject, np.where(np.isnan(run_start), moment, run_start), np.where(active, np.nan, run_start)
        )
        prev_time = moment
    return scores, rejected


def select_kalman_threshold(
    frame: pd.DataFrame, sigma_acc: float, sigma_meas: float, speed_limit: float, mcc_band: float
) -> dict[str, float]:
    """Подбирает порог NIS: сетка по NIS без замены и χ²₀.₉₉₉(2), прогон с заменой на каждого кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, n, y).
        sigma_acc: СКО ускорения, м/с².
        sigma_meas: СКО измерения, м.
        speed_limit: Предел скорости перезапуска, м/с.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    base_scores, _ = kalman_detect(frame, sigma_acc, sigma_meas, speed_limit, np.array([np.inf]))
    finite = base_scores[0][np.isfinite(base_scores[0])]
    candidates = np.unique(np.r_[threshold_candidates(finite), KALMAN_CHI999])
    scores, rejected = kalman_detect(frame, sigma_acc, sigma_meas, speed_limit, candidates)
    ground_truth_anomaly = frame["y"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


In [ ]:
HMM_STATES_PER_CLASS = 3
HMM_MAX_ITER = 200
HMM_TOL = 1e-4
HMM_RESTARTS = 5
# Нижняя граница дисперсии признака в режиме: без неё EM схлопывает режим на совпадающих точках.
HMM_VAR_MIN = 1e-3
# Нижняя граница начальных и переходных вероятностей: без неё отрезок, начинающийся в режиме, ни разу не встреченном
# в начале отрезка обучения, получает вероятность 0 и параметры EM становятся NaN.
HMM_PROB_MIN = 1e-6
# Граница «аномалия вероятнее штатного хода».
HMM_CANONICAL_THRESHOLD = 0.5


def _hmm_log_emission(features: np.ndarray, model: dict[str, np.ndarray]) -> np.ndarray:
    """Логарифм плотности признаков в каждом режиме (n, K), диагональная гауссовская плотность."""
    diff = features[:, None, :] - model["means"][None, :, :]
    var = model["vars"][None, :, :]
    return -0.5 * np.sum(diff * diff / var + np.log(2.0 * np.pi * var), axis=2)


def _hmm_runs(first: np.ndarray, scored: np.ndarray) -> list[np.ndarray]:
    """Позиции непрерывных участков точек с признаками внутри отрезка — по участку на перезапуск фильтра."""
    segment = np.cumsum(first)
    breaks = np.flatnonzero(np.diff(scored.astype(np.int8)) != 0) + 1
    breaks = np.union1d(breaks, np.flatnonzero(first))
    runs = np.split(np.arange(len(first)), breaks)
    return [run for run in runs if len(run) and scored[run[0]] and segment[run[0]] == segment[run[-1]]]


def fit_hmm_on_train(train: pd.DataFrame) -> dict[str, object]:
    """EM для HMM с режимами штатного хода и аномалии; в размеченной точке режимы чужого класса запрещены.

    Признаки — по последней точке штатного хода, нормировка — по штатному ходу, как у DBSCAN. Точка штатного хода
    после серии аномалий и следующая — без признаков, цепь не рвётся (режим без наблюдения). Из HMM_RESTARTS
    запусков с фиксированными seed выбирается лучший по логарифму правдоподобия.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.

    Returns:
        dict[str, object]: center, scale, startprob, transmat, means, vars (K, d), loglik, iterations, converged.
    """
    labels = train["y"].astype(str).to_numpy()
    index, first = dbscan_segments(train, np.ones(len(train), dtype=bool))
    move = labels[index] == "move"
    position = np.arange(len(index))
    last_move = np.maximum.accumulate(np.where(move | first, position, -1))
    # Точка штатного хода после серии аномалий — без предшественника: прыжок через серию не попадает в признаки.
    restart = move & ~first & ~np.r_[True, move[:-1]]
    before = np.where(first | restart, -1, np.r_[-1, last_move[:-1]])
    raw = dbscan_features(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], before)
    scored = ~np.isnan(raw[:, 0])
    # Она и следующая точка без признаков, но цепь не рвётся: в них режим без наблюдения.
    blank = (restart | np.r_[False, restart[:-1]]) & ~first & ~scored
    if not np.any(scored & move) or not np.any(scored & ~move):
        raise RuntimeError("Для HMM нужны точки штатного хода и аномалии на обучающей выборке.")
    center, scale = robust_params(raw[scored & move])
    features = apply_robust(raw, center, scale)
    runs = _hmm_runs(first, scored | blank)
    n_states = 2 * HMM_STATES_PER_CLASS
    anomaly_state = np.arange(n_states) >= HMM_STATES_PER_CLASS
    # Запрет режимов чужого класса: log 0 в логарифме плотности.
    forbid = np.where(move[:, None], np.where(anomaly_state, -np.inf, 0.0), np.where(anomaly_state, 0.0, -np.inf))
    best = None
    for restart in range(HMM_RESTARTS):
        rng = np.random.default_rng(SEED + restart)
        means = np.vstack(
            [
                features[rng.choice(np.flatnonzero(scored & move), HMM_STATES_PER_CLASS, replace=False)],
                features[rng.choice(np.flatnonzero(scored & ~move), HMM_STATES_PER_CLASS, replace=False)],
            ]
        )
        variances = np.vstack(
            [
                np.tile(np.maximum(features[scored & move].var(axis=0), HMM_VAR_MIN), (HMM_STATES_PER_CLASS, 1)),
                np.tile(np.maximum(features[scored & ~move].var(axis=0), HMM_VAR_MIN), (HMM_STATES_PER_CLASS, 1)),
            ]
        )
        model = {
            "startprob": np.full(n_states, 1.0 / n_states),
            "transmat": np.full((n_states, n_states), 0.1 / (n_states - 1))
            + np.eye(n_states) * (0.9 - 0.1 / (n_states - 1)),
            "means": means,
            "vars": variances,
        }
        previous, converged = -np.inf, False
        for iteration in range(1, HMM_MAX_ITER + 1):
            start_count = np.zeros(n_states)
            trans_count = np.zeros((n_states, n_states))
            weight = np.zeros(n_states)
            first_moment = np.zeros_like(means)
            second_moment = np.zeros_like(means)
            loglik = 0.0
            for run in runs:
                observed = scored[run]
                log_emission = np.zeros((len(run), n_states))
                log_emission[observed] = _hmm_log_emission(features[run][observed], model)
                log_frame = np.ascontiguousarray(log_emission + forbid[run])
                log_prob, forward = _hmmc.forward_log(model["startprob"], model["transmat"], log_frame)
                backward = _hmmc.backward_log(model["startprob"], model["transmat"], log_frame)
                posterior = np.exp(forward + backward - log_prob)
                loglik += log_prob
                start_count += posterior[0]
                if len(run) > 1:
                    # compute_log_xi_sum уже нормирует на правдоподобие участка.
                    trans_count += np.exp(_hmmc.compute_log_xi_sum(forward, model["transmat"], backward, log_frame))
                weight += posterior[observed].sum(axis=0)
                first_moment += posterior[observed].T @ features[run][observed]
                second_moment += posterior[observed].T @ features[run][observed] ** 2
            start_count += HMM_PROB_MIN
            model["startprob"] = start_count / start_count.sum()
            trans_count += HMM_PROB_MIN
            model["transmat"] = trans_count / trans_count.sum(axis=1, keepdims=True)
            model["means"] = first_moment / weight[:, None]
            model["vars"] = np.maximum(second_moment / weight[:, None] - model["means"] ** 2, HMM_VAR_MIN)
            if abs(loglik - previous) < HMM_TOL * abs(loglik):
                converged = True
                break
            previous = loglik
        if not converged:
            print(f"  предупреждение: EM HMM не сошёлся за {HMM_MAX_ITER} итераций (запуск {restart + 1})")
        if best is None or loglik > best["loglik"]:
            best = {**model, "loglik": loglik, "iterations": iteration, "converged": converged}
    return {"center": center, "scale": scale, **best}


def hmm_filter_scores(features: np.ndarray, first: np.ndarray, model: dict[str, object]) -> np.ndarray:
    """Причинная апостериорная вероятность режима аномалии; фильтр начинается заново на каждом участке.

    Args:
        features: Нормированные признаки (P, d), NaN — точка без признаков.
        first: Маска первой точки отрезка (P,).
        model: Результат fit_hmm_on_train.

    Returns:
        np.ndarray: Оценка (P,), NaN — точка без оценки.
    """
    scored = ~np.isnan(features[:, 0])
    result = np.full(len(features), np.nan)
    anomaly_state = np.arange(len(model["startprob"])) >= HMM_STATES_PER_CLASS
    for run in _hmm_runs(first, scored):
        log_frame = np.ascontiguousarray(_hmm_log_emission(features[run], model))
        _, forward = _hmmc.forward_log(model["startprob"], model["transmat"], log_frame)
        posterior = np.exp(forward - logsumexp(forward, axis=1, keepdims=True))
        result[run] = posterior[:, anomaly_state].sum(axis=1)
    return result


def hmm_detect(
    frame: pd.DataFrame, model: dict[str, object], taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Оценка HMM в два прохода, сразу для нескольких порогов.

    Второй проход — признаки по последним точкам, принятым первым проходом; фильтр пересчитывается целиком.

    Args:
        frame: DataFrame с колонками x, n и time.
        model: Результат fit_hmm_on_train.
        taus: Пороги; np.inf — без отвергнутых точек.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: оценка второго прохода (C, N), NaN — точка без оценки;
        маска отвергнутых (C, N); оценка первого прохода (N,).
    """
    index, first = dbscan_segments(frame, np.ones(len(frame), dtype=bool))
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    position = np.arange(len(index))
    first_features = apply_robust(
        dbscan_features(xy, time_sec, np.where(first, -1, position - 1)), model["center"], model["scale"]
    )
    first_scores = hmm_filter_scores(first_features, first, model)
    scored = ~np.isnan(first_scores)
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    scores = np.full((len(thresholds), len(frame)), np.nan)
    rejected = np.zeros((len(thresholds), len(frame)), dtype=bool)
    for row, tau in enumerate(thresholds):
        accepted = ~(first_scores > tau)
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(first, -1, np.r_[-1, last_accepted[:-1]])
        second_features = apply_robust(dbscan_features(xy, time_sec, before), model["center"], model["scale"])
        second_scores = hmm_filter_scores(second_features, first, model)
        scores[row, index[scored]] = second_scores[scored]
        rejected[row, index[scored]] = second_scores[scored] > tau
    first_full = np.full(len(frame), np.nan)
    first_full[index[scored]] = first_scores[scored]
    return scores, rejected, first_full


def select_hmm_threshold(frame: pd.DataFrame, model: dict[str, object], mcc_band: float) -> dict[str, float]:
    """Подбирает порог: общая сетка по оценке первого прохода и 0.5, два прохода на кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, n, y).
        model: Результат fit_hmm_on_train.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    _, _, first = hmm_detect(frame, model, np.array([np.inf]))
    candidates = np.unique(np.r_[threshold_candidates(first[np.isfinite(first)]), HMM_CANONICAL_THRESHOLD])
    scores, rejected, _ = hmm_detect(frame, model, candidates)
    ground_truth_anomaly = frame["y"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


In [ ]:
DBSCAN_MIN_SAMPLES = 6
DBSCAN_EPS_LEVELS = np.linspace(0.5, 0.999, 30)


def fit_dbscan_on_train(train: pd.DataFrame) -> dict[str, np.ndarray]:
    """Нормировка и расстояния до MinPts-го соседа (с самой точкой) по штатному ходу обучающей выборки.

    Признаки считаются по соседним точкам обучающих отрезков штатного хода (move_segments): отрезок рвётся на
    каждой аномальной точке и на разрыве по времени.

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.

    Returns:
        dict[str, np.ndarray]: center, scale — робастная нормировка; points — нормированные признаки штатного
        хода (M, 3); k_distance — расстояние до MinPts-го соседа (M,).
    """
    index, first = segments_layout(move_segments(train))
    before = np.where(first, -1, np.arange(len(index)) - 1)
    raw = dbscan_features(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], before)
    raw = raw[~np.isnan(raw[:, 0])]
    if len(raw) <= DBSCAN_MIN_SAMPLES:
        raise RuntimeError("Мало точек штатного хода для DBSCAN.")
    center, scale = robust_params(raw)
    points = apply_robust(raw, center, scale)
    neighbors = NearestNeighbors(n_neighbors=DBSCAN_MIN_SAMPLES).fit(points)
    k_distance = neighbors.kneighbors(points)[0][:, -1]
    return {"center": center, "scale": scale, "points": points, "k_distance": k_distance}


def dbscan_detect(
    frame: pd.DataFrame, model: dict[str, np.ndarray], eps_values: np.ndarray
) -> tuple[np.ndarray, np.ndarray]:
    """Расстояние до ближайшей основной точки в два прохода, сразу для нескольких ε.

    Основные точки при данном ε — точки штатного хода с расстоянием до MinPts-го соседа не больше ε. Первый
    проход — признаки по предыдущим точкам отрезка; второй — по последним точкам, принятым первым проходом.

    Args:
        frame: DataFrame с колонками x, n и time.
        model: Результат fit_dbscan_on_train.
        eps_values: Радиусы ε в нормированных признаках.

    Returns:
        tuple[np.ndarray, np.ndarray]: оценка аномальности (C, N), NaN — точка без оценки; маска отвергнутых.
    """
    index, first = dbscan_segments(frame, np.ones(len(frame), dtype=bool))
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    position = np.arange(len(index))
    radii = np.atleast_1d(np.asarray(eps_values, dtype=float))
    first_features = apply_robust(
        dbscan_features(xy, time_sec, np.where(first, -1, position - 1)), model["center"], model["scale"]
    )
    scored = ~np.isnan(first_features[:, 0])
    scores = np.full((len(radii), len(frame)), np.nan)
    rejected = np.zeros((len(radii), len(frame)), dtype=bool)
    for row, eps in enumerate(radii):
        core = model["points"][model["k_distance"] <= eps]
        if len(core) == 0:
            raise RuntimeError(f"Нет основных точек DBSCAN при ε = {eps:.4g}.")
        nearest = NearestNeighbors(n_neighbors=1).fit(core)

        def distance(features: np.ndarray) -> np.ndarray:
            return nearest.kneighbors(features[scored])[0][:, 0]

        accepted = np.ones(len(index), dtype=bool)
        accepted[scored] = distance(first_features) <= eps
        # Последняя принятая точка до текущей; первые точки отрезка без оценки и приняты — отрезки не смешиваются.
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(first, -1, np.r_[-1, last_accepted[:-1]])
        second = apply_robust(dbscan_features(xy, time_sec, before), model["center"], model["scale"])
        second_scores = distance(second)
        scores[row, index[scored]] = second_scores
        rejected[row, index[scored]] = second_scores > eps
    return scores, rejected


def select_dbscan_eps(frame: pd.DataFrame, model: dict[str, np.ndarray], mcc_band: float) -> dict[str, float]:
    """Подбирает ε по квантилям расстояния до MinPts-го соседа: максимум MCC, в полосе — минимум DL.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, n, y).
        model: Результат fit_dbscan_on_train.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau (выбранное ε), train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    candidates = np.unique(np.quantile(model["k_distance"], DBSCAN_EPS_LEVELS))
    scores, rejected = dbscan_detect(frame, model, candidates)
    ground_truth_anomaly = frame["y"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(eps), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, eps in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


In [ ]:
IF_N_ESTIMATORS = 200
IF_MAX_SAMPLES = 256
# Граница аномальности из определения оценки: s > 0.5 — путь короче среднего в двоичном дереве поиска.
IF_CANONICAL_THRESHOLD = 0.5


def fit_isolation_forest_on_train(train: pd.DataFrame) -> dict[str, object]:
    """Лес по нормированным признакам штатного хода обучающей выборки.

    Признаки и нормировка — как у DBSCAN: по обучающим отрезкам штатного хода (move_segments).

    Args:
        train: Обучающий DataFrame с колонками x, n, time и y.

    Returns:
        dict[str, object]: center, scale — робастная нормировка; points — набор T (M, 3); forest — обученный лес.
    """
    index, first = segments_layout(move_segments(train))
    before = np.where(first, -1, np.arange(len(index)) - 1)
    raw = dbscan_features(enu_xy_from_frame(train)[index], time_seconds_from_frame(train)[index], before)
    raw = raw[~np.isnan(raw[:, 0])]
    if len(raw) < 2:
        raise RuntimeError("Мало точек штатного хода для Isolation Forest.")
    center, scale = robust_params(raw)
    points = apply_robust(raw, center, scale)
    forest = IsolationForest(
        n_estimators=IF_N_ESTIMATORS, max_samples=min(IF_MAX_SAMPLES, len(points)), random_state=SEED, n_jobs=-1
    ).fit(points)
    return {"center": center, "scale": scale, "points": points, "forest": forest}


def isolation_forest_detect(
    frame: pd.DataFrame, model: dict[str, object], taus: np.ndarray
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Оценка s = 2^(-h̄/c) в два прохода, сразу для нескольких порогов.

    Второй проход — признаки по последним точкам, принятым первым проходом; оценка пересчитывается только у точек,
    признаки которых изменились, и только для ещё не встречавшихся признаков (cached_rows).

    Args:
        frame: DataFrame с колонками x, n и time.
        model: Результат fit_isolation_forest_on_train.
        taus: Пороги; np.inf — без отвергнутых точек.

    Returns:
        tuple[np.ndarray, np.ndarray, np.ndarray]: оценка второго прохода (C, N), NaN — точка без оценки;
        маска отвергнутых (C, N); оценка первого прохода (N,).
    """
    index, first = dbscan_segments(frame, np.ones(len(frame), dtype=bool))
    xy = enu_xy_from_frame(frame)[index]
    time_sec = time_seconds_from_frame(frame)[index]
    position = np.arange(len(index))
    first_features = apply_robust(
        dbscan_features(xy, time_sec, np.where(first, -1, position - 1)), model["center"], model["scale"]
    )
    scored = ~np.isnan(first_features[:, 0])
    first_scores = np.full(len(index), np.nan)
    # score_samples в sklearn — оценка статьи со знаком минус.
    first_scores[scored] = -model["forest"].score_samples(first_features[scored])
    thresholds = np.atleast_1d(np.asarray(taus, dtype=float))
    scores = np.full((len(thresholds), len(frame)), np.nan)
    rejected = np.zeros((len(thresholds), len(frame)), dtype=bool)
    # Оценки второго прохода повторяются при разных порогах: кэш по ключу признаков.
    cache: dict[bytes, object] = {}
    for row, tau in enumerate(thresholds):
        accepted = ~(first_scores > tau)
        last_accepted = np.maximum.accumulate(np.where(accepted, position, -1))
        before = np.where(first, -1, np.r_[-1, last_accepted[:-1]])
        second_features = apply_robust(dbscan_features(xy, time_sec, before), model["center"], model["scale"])
        second_scores = first_scores.copy()
        changed = scored & np.any(second_features != first_features, axis=1)
        if np.any(changed):
            changed_rows = np.flatnonzero(changed)
            second_scores[changed] = cached_rows(
                feature_keys(before, changed_rows),
                cache,
                lambda missing: -model["forest"].score_samples(second_features[changed_rows[missing]]),
            )
        scores[row, index[scored]] = second_scores[scored]
        rejected[row, index[scored]] = second_scores[scored] > tau
    first_full = np.full(len(frame), np.nan)
    first_full[index[scored]] = first_scores[scored]
    return scores, rejected, first_full


def select_isolation_forest_threshold(
    frame: pd.DataFrame, model: dict[str, object], mcc_band: float
) -> dict[str, float]:
    """Подбирает порог: общая сетка по оценке первого прохода и 0.5, два прохода на кандидата.

    Args:
        frame: Обучающий DataFrame (time, lat, lon, x, n, y).
        model: Результат fit_isolation_forest_on_train.
        mcc_band: Допуск по MCC относительно лучшего значения.

    Returns:
        dict[str, float]: tau, train_mcc, train_f1, train_dl_m, best_mcc, band_size.
    """
    _, _, first = isolation_forest_detect(frame, model, np.array([np.inf]))
    candidates = np.unique(np.r_[threshold_candidates(first[np.isfinite(first)]), IF_CANONICAL_THRESHOLD])
    scores, rejected, _ = isolation_forest_detect(frame, model, candidates)
    ground_truth_anomaly = frame["y"].astype(str).to_numpy() == "anomaly"
    records = [
        _threshold_record(float(tau), rejected[row], frame, ground_truth_anomaly, ~np.isnan(scores[row]))
        for row, tau in enumerate(candidates)
    ]
    return select_threshold_from_records(records, mcc_band)


## Поиск блока

Test — test сценария df1 ($\rho=0.6$); пул — вся обучающая выборка до него. Блок выбирается только по пулу, test в выборе не участвует.

Требования к блоку, без обучения моделей:

1. окон по 16 подряд идущих точек обучающих отрезков — не меньше, чем нужно нейросетям: 8000 обучающих и калибровочный блок (15%);
2. эпизодов аномалий — не меньше 20: выбор порога и HMM;
3. распределение признаков $(v, a^{\mathrm{long}}, a^{\mathrm{lat}})$ штатного хода — как в пуле: наибольшая по признакам статистика Колмогорова — Смирнова не больше 0.05;
4. распределение признаков аномальных точек — как в пуле: та же статистика не больше 0.10 (аномальных точек меньше, и они разнороднее). Аномалии нужны выбору порога у всех моделей и обучению HMM;
5. покрытие режимов движения: не меньше 95% случайных точек штатного хода пула имеют в блоке соседа в пространстве нормированных признаков не дальше 95-го перцентиля расстояний до ближайшего соседа внутри пула.

Длины блоков — от 5 до 100% пула с шагом 5%, начала — с шагом 2.5% пула. Выбирается самая короткая длина, при которой проходит хотя бы один блок, из прошедших — с наименьшей статистикой Колмогорова — Смирнова. В таблице — все блоки выбранной длины.

In [ ]:
TS_KS_MAX = 0.05
# Аномальных точек с признаками меньше и они разнороднее, чем штатный ход: порог мягче.
TS_KS_ANOMALY_MAX = 0.10
# Покрытие: доля точек пула, у которых в блоке есть сосед не дальше TS_COVERAGE_QUANTILE-квантиля расстояний до
# ближайшего соседа внутри пула.
TS_COVERAGE_MIN = 0.95
TS_COVERAGE_QUANTILE = 0.95
TS_COVERAGE_SAMPLE = 5_000
TS_MIN_EPISODES = 20
# Окно нейросетей и Robust PCA, точки; окон нужно на N_WINDOWS обучающих и калибровочный блок.
TS_WINDOW = 16
TS_MIN_WINDOWS = int(np.ceil(N_WINDOWS / (1.0 - NN_CALIB_FRACTION)))
TS_LENGTHS = np.round(np.arange(0.05, 1.0001, 0.05), 2)
TS_START_STEP = 0.025


def ks_distance(sample: np.ndarray, reference_sorted: np.ndarray) -> float:
    """Статистика Колмогорова — Смирнова: наибольшая разность эмпирических функций распределения.

    Args:
        sample: Выборка.
        reference_sorted: Опорная выборка по возрастанию.

    Returns:
        float: Расстояние от 0 до 1.
    """
    ordered = np.sort(sample)
    grid = np.r_[ordered, reference_sorted]
    own = np.searchsorted(ordered, grid, side="right") / len(ordered)
    reference = np.searchsorted(reference_sorted, grid, side="right") / len(reference_sorted)
    return float(np.max(np.abs(own - reference)))


def anomaly_runs(frame: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Эпизоды аномалий: первая и последняя строки каждой серии подряд идущих аномальных точек.

    Args:
        frame: DataFrame с колонкой y.

    Returns:
        tuple[np.ndarray, np.ndarray]: начала и концы эпизодов.
    """
    anomaly = np.r_[False, frame["y"].astype(str).to_numpy() == "anomaly", False].astype(np.int8)
    change = np.diff(anomaly)
    return np.flatnonzero(change == 1), np.flatnonzero(change == -1) - 1


def pool_layout(pool: pd.DataFrame) -> dict[str, np.ndarray]:
    """Признаки, окна и эпизоды пула с границами строк, от которых они зависят.

    Признак точки штатного хода зависит от неё и двух предыдущих точек обучающего отрезка, окно — от своих точек
    и двух предшественников первой; признак аномальной точки — от неё и двух предыдущих точек ряда, как в первом
    проходе детекторов. В блоке [s, e) признак (окно) сохраняется, если все эти строки внутри блока.

    Args:
        pool: Пул обучения — ряд до test.

    Returns:
        dict[str, np.ndarray]: features (P, 3) и их строки from/to; scaled — робастная нормировка по пулу;
        anomaly_features и их строки from/to; window_from, window_to; episode_start, episode_end.
    """
    index, first = segments_layout(move_segments(pool))
    position = np.arange(len(index))
    raw = dbscan_features(
        enu_xy_from_frame(pool)[index], time_seconds_from_frame(pool)[index], np.where(first, -1, position - 1)
    )
    valid = ~np.isnan(raw[:, 0])
    origin = index[np.maximum(position - 2, 0)]
    starts = rpca_window_starts(first, valid, TS_WINDOW)
    center, scale = robust_params(raw[valid])
    episode_start, episode_end = anomaly_runs(pool)
    all_index, all_first = dbscan_segments(pool, np.ones(len(pool), dtype=bool))
    all_position = np.arange(len(all_index))
    all_raw = dbscan_features(
        enu_xy_from_frame(pool)[all_index],
        time_seconds_from_frame(pool)[all_index],
        np.where(all_first, -1, all_position - 1),
    )
    anomaly = (pool["y"].astype(str).to_numpy()[all_index] == "anomaly") & ~np.isnan(all_raw[:, 0])
    return {
        "anomaly_features": all_raw[anomaly],
        "anomaly_from": all_index[np.maximum(all_position - 2, 0)][anomaly],
        "anomaly_to": all_index[anomaly],
        "features": raw[valid],
        "scaled": apply_robust(raw[valid], center, scale),
        "feature_from": origin[valid],
        "feature_to": index[valid],
        "window_from": origin[starts],
        "window_to": index[starts + TS_WINDOW - 1],
        "episode_start": episode_start,
        "episode_end": episode_end,
    }


def block_check(layout: dict[str, np.ndarray], reference: dict[str, object], start: int, stop: int) -> dict[str, object]:
    """Требования к блоку [start, stop) пула: окна, эпизоды, распределения признаков штатного хода и аномалий,
    покрытие режимов.

    Покрытие считается, только если выполнены остальные требования: это самая дорогая проверка.

    Args:
        layout: Результат pool_layout.
        reference: Опорные величины пула: sorted, anomaly_sorted — признаки штатного хода и аномалий по
            возрастанию, sample — точки для покрытия, radius — радиус покрытия.
        start: Первая строка блока.
        stop: Строка после последней.

    Returns:
        dict[str, object]: показатели блока и признак прохождения.
    """
    inside = (layout["feature_from"] >= start) & (layout["feature_to"] < stop)
    windows = int(np.sum((layout["window_from"] >= start) & (layout["window_to"] < stop)))
    episodes = int(np.sum((layout["episode_start"] >= start) & (layout["episode_end"] < stop)))
    anomaly_inside = (layout["anomaly_from"] >= start) & (layout["anomaly_to"] < stop)
    result = {"Начало": start, "Точек": stop - start, "Признаков": int(inside.sum()), "Окон": windows,
              "Эпизодов": episodes, "KS": np.nan, "KS аномалий": np.nan, "Покрытие": np.nan}
    if inside.sum() < 2 or anomaly_inside.sum() < 2:
        result["Проходит"] = False
        return result
    result["KS"] = max(
        ks_distance(layout["features"][inside, column], reference["sorted"][column]) for column in range(3)
    )
    result["KS аномалий"] = max(
        ks_distance(layout["anomaly_features"][anomaly_inside, column], reference["anomaly_sorted"][column])
        for column in range(3)
    )
    cheap = (
        windows >= TS_MIN_WINDOWS
        and episodes >= TS_MIN_EPISODES
        and result["KS"] <= TS_KS_MAX
        and result["KS аномалий"] <= TS_KS_ANOMALY_MAX
    )
    if cheap:
        distance = NearestNeighbors(n_neighbors=1).fit(layout["scaled"][inside]).kneighbors(reference["sample"])[0]
        result["Покрытие"] = float(np.mean(distance[:, 0] <= reference["radius"]))
    result["Проходит"] = bool(cheap and result["Покрытие"] >= TS_COVERAGE_MIN)
    return result


ts_pool, ts_test = df1["train"], df1["test"]
ts_layout = pool_layout(ts_pool)
sample_rows = np.random.default_rng(SEED).choice(
    len(ts_layout["scaled"]), min(TS_COVERAGE_SAMPLE, len(ts_layout["scaled"])), replace=False
)
pool_distance = NearestNeighbors(n_neighbors=2).fit(ts_layout["scaled"]).kneighbors(ts_layout["scaled"][sample_rows])[0]
ts_reference = {
    "sorted": [np.sort(ts_layout["features"][:, column]) for column in range(3)],
    "anomaly_sorted": [np.sort(ts_layout["anomaly_features"][:, column]) for column in range(3)],
    "sample": ts_layout["scaled"][sample_rows],
    "radius": float(np.quantile(pool_distance[:, 1], TS_COVERAGE_QUANTILE)),
}
print(
    f"Пул train: {nfmt(len(ts_pool))} точек, признаков штатного хода {nfmt(len(ts_layout['features']))}, "
    f"окон {nfmt(len(ts_layout['window_from']))}, эпизодов аномалий {nfmt(len(ts_layout['episode_start']))}; "
    f"test: {nfmt(len(ts_test))} точек"
)
print(
    f"Требования: окон ≥ {nfmt(TS_MIN_WINDOWS)}, эпизодов ≥ {TS_MIN_EPISODES}, KS ≤ {TS_KS_MAX}, "
    f"KS аномалий ≤ {TS_KS_ANOMALY_MAX}, "
    f"покрытие ≥ {TS_COVERAGE_MIN:.0%} (радиус {ts_reference['radius']:.3f})"
)
ts_rows: list[dict[str, object]] = []
ts_block_bounds = None
for length in TS_LENGTHS:
    size = int(length * len(ts_pool))
    count = int(round((1.0 - length) / TS_START_STEP)) + 1
    for start in np.unique(np.linspace(0, len(ts_pool) - size, count).astype(int)):
        ts_rows.append({"Доля": length, **block_check(ts_layout, ts_reference, int(start), int(start) + size)})
    current = [row for row in ts_rows if row["Доля"] == length]
    passing = [row for row in current if row["Проходит"]]
    best = min(passing, key=lambda row: row["KS"]) if passing else None
    if best is not None:
        ts_block_bounds = (best["Начало"], best["Начало"] + best["Точек"])
        print(f"Доля {length:.0%}: проходит {len(passing)} блоков; выбран с началом {nfmt(best['Начало'])}")
        break
    failed = {
        "окна": sum(row["Окон"] < TS_MIN_WINDOWS for row in current),
        "эпизоды": sum(row["Эпизодов"] < TS_MIN_EPISODES for row in current),
        "KS": sum(not row["KS"] <= TS_KS_MAX for row in current),
        "KS аномалий": sum(not row["KS аномалий"] <= TS_KS_ANOMALY_MAX for row in current),
        "покрытие": sum(row["Покрытие"] < TS_COVERAGE_MIN for row in current),
    }
    reasons = ", ".join(f"{name} {count}" for name, count in failed.items() if count)
    print(f"Доля {length:.0%}: ни один из {len(current)} блоков не проходит; не выполнено — {reasons}")
ts_blocks = pd.DataFrame(ts_rows)
if ts_block_bounds is None:
    raise RuntimeError("Ни один блок пула не проходит требования; ослабьте TS_KS_MAX или TS_COVERAGE_MIN.")
ts_block = ts_pool.iloc[ts_block_bounds[0] : ts_block_bounds[1]].reset_index(drop=True)
display(ts_blocks[ts_blocks["Доля"] == ts_blocks["Доля"].max()].round(4))

## Проверка на представителях

Каждый представитель обучается и выбирает порог дважды — на всём пуле и на блоке — и размечает один и тот же test. Неопределённость MCC — 95% интервал блочного бутстрепа (блоки по 1000 точек, 200 повторов). Блок годится для модели, если её MCC на блоке не ниже нижней границы интервала MCC при обучении на пуле.

In [ ]:
TS_BOOTSTRAP = 200
# Точек в блоке бутстрепа: соседние точки зависимы, блоки сохраняют эту зависимость.
TS_BOOT_BLOCK = 1_000


def run_kalman(train: pd.DataFrame, test: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Фильтр Калмана: обучение и порог на train, разметка test (маска anomaly, маска точек с оценкой)."""
    sigma_acc, sigma_meas = fit_kalman_on_train(train)
    speed_limit = restart_speed_limit(train)
    selection = select_kalman_threshold(train, sigma_acc, sigma_meas, speed_limit, MCC_BAND)
    scores, rejected = kalman_detect(test, sigma_acc, sigma_meas, speed_limit, np.array([selection["tau"]]))
    return rejected[0], ~np.isnan(scores[0])


def run_isolation_forest(train: pd.DataFrame, test: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """Isolation Forest: обучение и порог на train, разметка test (маска anomaly, маска точек с оценкой)."""
    model = fit_isolation_forest_on_train(train)
    selection = select_isolation_forest_threshold(train, model, MCC_BAND)
    scores, rejected, _ = isolation_forest_detect(test, model, np.array([selection["tau"]]))
    return rejected[0], ~np.isnan(scores[0])


def run_dbscan(train: pd.DataFrame, test: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """DBSCAN: обучение и ε на train, разметка test (маска anomaly, маска точек с оценкой)."""
    model = fit_dbscan_on_train(train)
    selection = select_dbscan_eps(train, model, MCC_BAND)
    scores, rejected = dbscan_detect(test, model, np.array([selection["tau"]]))
    return rejected[0], ~np.isnan(scores[0])


def run_hmm(train: pd.DataFrame, test: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """HMM: обучение и порог на train, разметка test (маска anomaly, маска точек с оценкой)."""
    model = fit_hmm_on_train(train)
    selection = select_hmm_threshold(train, model, MCC_BAND)
    scores, rejected, _ = hmm_detect(test, model, np.array([selection["tau"]]))
    return rejected[0], ~np.isnan(scores[0])


def run_mpct(train: pd.DataFrame, test: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    """MPCT: v_+ на train, разметка test (маска anomaly, маска точек с решением)."""
    selection = select_mpct_speed(train, MCC_BAND)
    rejected, scored = mpct_detect(test, np.array([float(selection["tau"])]))
    return rejected[0], scored


# Представители семейств: прогноз, новизна по признакам, плотность, оба класса, без обучения параметров.
TS_MODELS = (
    ("Фильтр Калмана", run_kalman),
    ("Isolation Forest", run_isolation_forest),
    ("DBSCAN", run_dbscan),
    ("HMM", run_hmm),
    ("MPCT", run_mpct),
)


def bootstrap_interval(
    test: pd.DataFrame, predicted: np.ndarray, scored: np.ndarray, rng: np.random.Generator
) -> dict[str, np.ndarray]:
    """95% интервалы F1 и MCC на test блочным бутстрепом по точкам с оценкой.

    Args:
        test: Тестовый DataFrame с колонкой y.
        predicted: Маска предсказанных anomaly.
        scored: Маска точек с оценкой.
        rng: Генератор случайных чисел.

    Returns:
        dict[str, np.ndarray]: f1, mcc — границы 2.5% и 97.5%.
    """
    truth = test["y"].astype(str).to_numpy() == "anomaly"
    block = np.arange(len(test)) // TS_BOOT_BLOCK
    n_blocks = int(block.max()) + 1
    counts = np.zeros((n_blocks, 4))
    for column, mask in enumerate((predicted & truth, predicted & ~truth, ~predicted & truth, ~predicted & ~truth)):
        counts[:, column] = np.bincount(block[scored & mask], minlength=n_blocks)
    tp, fp, fn, tn = counts[rng.integers(0, n_blocks, (TS_BOOTSTRAP, n_blocks))].sum(axis=1).T
    f1 = 2.0 * tp / np.maximum(2.0 * tp + fp + fn, 1.0)
    mcc = (tp * tn - fp * fn) / np.sqrt(np.maximum((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn), 1e-12))
    return {"f1": np.percentile(f1, [2.5, 97.5]), "mcc": np.percentile(mcc, [2.5, 97.5])}


ts_rng = np.random.default_rng(SEED)
ts_results: list[dict[str, object]] = []
print(f"Блок: строки {nfmt(ts_block_bounds[0])}–{nfmt(ts_block_bounds[1] - 1)} пула, {nfmt(len(ts_block))} точек")
for model_name, run_model in TS_MODELS:
    for train_name, train in (("пул", ts_pool), ("блок", ts_block)):
        started = time.perf_counter()
        predicted, scored = run_model(train, ts_test)
        elapsed = time.perf_counter() - started
        metrics = evaluate_test(ts_test, predicted, scored)
        interval = bootstrap_interval(ts_test, predicted, scored, ts_rng)
        ts_results.append(
            {
                "Модель": model_name,
                "Обучение": train_name,
                "F1": metrics["f1"],
                "MCC": metrics["mcc"],
                "MCC 2.5%": interval["mcc"][0],
                "MCC 97.5%": interval["mcc"][1],
                "Precision": metrics["precision"],
                "Recall": metrics["recall"],
                "Время, с": elapsed,
            }
        )
        print(
            f"{model_name}, {train_name}: F1 {metrics['f1']:.4f}, MCC {metrics['mcc']:.4f} "
            f"[{interval['mcc'][0]:.3f}; {interval['mcc'][1]:.3f}] ({elapsed:.1f} с)"
        )
ts_results = pd.DataFrame(ts_results)
# Блок годится для модели, если её MCC на блоке попадает в интервал MCC при обучении на всём пуле.
ts_verdict = ts_results.pivot(index="Модель", columns="Обучение", values="MCC")
interval_pool = ts_results[ts_results["Обучение"] == "пул"].set_index("Модель")
ts_verdict["MCC пул 2.5%"] = interval_pool["MCC 2.5%"]
ts_verdict["MCC пул 97.5%"] = interval_pool["MCC 97.5%"]
ts_verdict["Годится"] = ts_verdict["блок"] >= ts_verdict["MCC пул 2.5%"]
time_pool = ts_results[ts_results["Обучение"] == "пул"]["Время, с"].sum()
time_block = ts_results[ts_results["Обучение"] == "блок"]["Время, с"].sum()
print(f"Время пяти моделей: пул {time_pool:.0f} с, блок {time_block:.0f} с")
display(ts_verdict.round(4))